# MePRAM preprocessing overview

This documented copy outlines how raw MePRAM tables are loaded, harmonised, and merged into modelling-ready datasets, highlighting the intent of each processing stage.

## Environment setup

Import the scientific Python stack, profiling utilities, and Spark placeholders required throughout the preprocessing pipeline.

In [1]:
import sqlite3
from ydata_profiling import ProfileReport
import pandas as pd
import numpy as np
import re
import os
import copy

## Run metadata and storage paths

Capture the execution timestamp and configure local folders / SQLite filenames used to persist intermediate outputs.

In [ ]:
today = pd.Timestamp("today").strftime("%Y%m%d_%H%M%S")
save_location = f"/data/ucct/bi/research/20260330_MEPRAM-RESULTS_CNM_C/ANALYSIS/01-preprocess/{today}_preprocess"
db_path = "../../../RAW/db_mepram_sepsis_vf.sqlite3"

### Defined functions used in the script

In [ ]:
def remove_outliers_and_categorise(df, variables):
    """
    This function replaces outliers with NaN using IQR and recodes quantitative variables into ranges 0-3.

    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    variables (list): List of column names to process.

    Returns:
    pd.DataFrame: The DataFrame with outliers replaced by NaN and new recoded columns.
    """

    df_processed = df.copy()
    
    for var in variables:
        if var not in df.columns:
            print(f"Variable '{var}' not found in the DataFrame. It will be skipped.")
            continue

        Q1 = df_processed[var].quantile(0.25)
        Q3 = df_processed[var].quantile(0.75)
        IQR = Q3 - Q1

        lower_limit = Q1 - 3 * IQR
        upper_limit = Q3 + 3 * IQR

        outliers_mask = (df_processed[var] < lower_limit) | (df_processed[var] > upper_limit)
        outliers_count = outliers_mask.sum()
        if outliers_count > 0:
            print(f"{outliers_count} outliers replaced with NaN in the variable '{var}'.")
            print(f"{df_processed.loc[outliers_mask, var]}")            

        df_processed.loc[outliers_mask, var] = np.nan

        new_column = f"{var}_recoded"

        df_processed[new_column] = pd.qcut(df_processed[var], 
                                           q=4, 
                                           labels=[0, 1, 2, 3], 
                                           duplicates='drop')

        df_processed[new_column] = df_processed[new_column].astype('Int64') 
        
        print(f"Variable '{var}' recoded in the column '{new_column}'.")
    
    return df_processed

def get_bacteria(row):
    """Return a comma-separated string of all organism columns that are positive (value == 1)."""
    return ", ".join([col for col in row.index if row[col] == 1])

def pick_dominant(row):
    """
    Choose the dominant organism from a combined count row:
    - If any organism has count ≥ 2, it is dominant (confirmed by multiple culture sources).
    - If exactly one organism has count == 1, return it.
    - Otherwise (e.g. only NEGATIVE present or ties with 1), return NEGATIVE.
    """
    if (row >= 2).any():
        return row.idxmax()          # organism confirmed across multiple culture types
    elif (row == 1).sum() == 1:
        return row[row == 1].index[0]  # single organism present in one culture type
    else:
        return "NEGATIVE"
    
def merge_columns(df_to_clean, column_list, new_col_name):
    """Sum a list of columns into a single aggregated column and drop the originals."""
    df_to_clean[new_col_name] = df_to_clean[column_list].sum(axis=1).astype(int)
    clean_df = df_to_clean.drop(columns=column_list)
    return clean_df
    
def dedupe_labels(labels):
    """Remove duplicate resistance phenotype labels and sort for consistent tuple representation."""
    return sorted(set(labels))

## Load database tables

Connect to the configured SQLite database and pull every `tbl_*` table into the `dataframes` dictionary for downstream transformations.

In [3]:
con = sqlite3.connect(db_path)

cursor = con.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

"""For older versions of the dataset
with open(os.path.join(save_location, "tbl_personid2center.sql"), "r") as f:
    sql_script = f.read()
    cursor.executescript(sql_script)

with open(os.path.join(save_location, "tbl_microorganismos.sql"), "r") as f:
    sql_script = f.read()
    cursor.executescript(sql_script)"""

tables = [row[0] for row in cursor.fetchall()]

print(tables)

dataframes = {}
for table in tables:
    print(f"Cargando la tabla: {table}")
    dataframes[table] = pd.read_sql_query(f"SELECT * FROM {table}", con)
con.close()

# Load tbl_codes2names for name mapping
tbl_codes2names = dataframes["tbl_codes2names"]

['tbl_master', 'tbl_paciente', 'tbl_comorbilidad', 'tbl_factores_riesgo_bmr', 'tbl_sintomas', 'tbl_signos', 'tbl_sepsis', 'tbl_infecciones_previas', 'tbl_colonizaciones_previas', 'tbl_tratamiento_antibiotico_previo', 'tbl_tratamiento_empirico', 'tbl_hemocultivo_de_urgencias', 'tbl_otros_cultivos_en_urgencias', 'tbl_codes2names', 'tbl_personid2center', 'tbl_microorganismos']
Cargando la tabla: tbl_master
Cargando la tabla: tbl_paciente
Cargando la tabla: tbl_comorbilidad
Cargando la tabla: tbl_factores_riesgo_bmr
Cargando la tabla: tbl_sintomas
Cargando la tabla: tbl_signos
Cargando la tabla: tbl_sepsis
Cargando la tabla: tbl_infecciones_previas
Cargando la tabla: tbl_colonizaciones_previas
Cargando la tabla: tbl_tratamiento_antibiotico_previo
Cargando la tabla: tbl_tratamiento_empirico
Cargando la tabla: tbl_hemocultivo_de_urgencias
Cargando la tabla: tbl_otros_cultivos_en_urgencias
Cargando la tabla: tbl_codes2names
Cargando la tabla: tbl_personid2center
Cargando la tabla: tbl_microor

## Patient master record (`tbl_paciente`)

Combine base patient information with centre identifiers and derive flags such as previous infection and resistant organism history.

In [5]:
df_pacientes = dataframes["tbl_paciente"].merge(dataframes["tbl_personid2center"], how="left")

# If present patient is present in tbl_infecciones_previas, inf_previa_sino = 1, else 0
df_pacientes["inf_previa_sino"] = np.where(np.isin(df_pacientes["person_id"].values, dataframes['tbl_infecciones_previas']["person_id"].values) == True, 1, 0)

# Create a new column "bmr_infec_previa" in df_pacientes that is 1 if the patient has any previous infection with BMR in tbl_infecciones_previas (column bmr_infec_previa), and 0 otherwise
bmr_previa = dataframes['tbl_infecciones_previas'].groupby("person_id")["bmr_infec_previa"].apply(lambda x: (x > 0).any())

df_pacientes = df_pacientes.merge(bmr_previa, on="person_id", how="left").fillna(False)
df_pacientes["bmr_infec_previa"] = np.where(df_pacientes["bmr_infec_previa"], 1, 0)
print(df_pacientes.head())

   person_id fecha_ingreso_urgencias fecha_nacimiento  edad  sexo  \
0          1              2021-04-22       1939-07-23    81     0   
1          2              2021-05-07       1939-07-23    81     0   
2          3              2021-06-04       1939-07-23    81     0   
3          4              2021-06-22       1939-07-23    81     0   
4          5              2021-03-24       1944-08-20    76     1   

   codigo_postal mujer_gestante  mayor_65  paciente_residencia  \
0           8036          False         1                    0   
1           8036          False         1                    0   
2           8036          False         1                    0   
3           8036          False         1                    0   
4           8036            0.0         1                    0   

           center    dag  inf_previa_sino  bmr_infec_previa  
0  CLINIC-IDIBAPS  621.0                0                 0  
1  CLINIC-IDIBAPS  621.0                0                 0  
2 

/tmp/ipykernel_1881424/1909595921.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pacientes = df_pacientes.merge(bmr_previa, on="person_id", how="left").fillna(False)


### Inspect pregnancy flag distribution

Exploratory cell to verify the `mujer_gestante` (pregnant woman) flag is encoded as expected.

In [6]:
df_pacientes["mujer_gestante"] = np.where(df_pacientes["mujer_gestante"], 1, 0)

print(df_pacientes.head())

   person_id fecha_ingreso_urgencias fecha_nacimiento  edad  sexo  \
0          1              2021-04-22       1939-07-23    81     0   
1          2              2021-05-07       1939-07-23    81     0   
2          3              2021-06-04       1939-07-23    81     0   
3          4              2021-06-22       1939-07-23    81     0   
4          5              2021-03-24       1944-08-20    76     1   

   codigo_postal  mujer_gestante  mayor_65  paciente_residencia  \
0           8036               0         1                    0   
1           8036               0         1                    0   
2           8036               0         1                    0   
3           8036               0         1                    0   
4           8036               0         1                    0   

           center    dag  inf_previa_sino  bmr_infec_previa  
0  CLINIC-IDIBAPS  621.0                0                 0  
1  CLINIC-IDIBAPS  621.0                0                 

## Comorbidity features (`tbl_comorbilidad`)

One-hot encode cancer and hepatopathy categories to expose chronic-condition indicators for modelling.

In [7]:
tbl_comorbilidad = dataframes['tbl_comorbilidad']
print(tbl_comorbilidad.shape)
tbl_comorbilidad = pd.get_dummies(tbl_comorbilidad, columns=["tipo_cancer","tipo_hepatopatia"])
print(tbl_comorbilidad)

(3913, 24)
      person_id fecha_ingreso_urgencias  infarto  insuficiencia_cardiaca  evp  \
0             1              2021-04-22        0                       0    0   
1             2              2021-05-07        0                       0    0   
2             3              2021-06-04        0                       0    0   
3             4              2021-06-22        0                       0    0   
4             5              2021-03-24        0                       0    0   
...         ...                     ...      ...                     ...  ...   
3908       3909              2023-01-18        0                       1    0   
3909       3910              2023-04-23        0                       0    0   
3910       3911              2023-04-10        0                       0    0   
3911       3912              2023-06-19        0                       0    0   
3912       3913              2023-06-26        0                       0    0   

      e_cerebrov

## BMR risk factor tracker (`tbl_factores_riesgo_bmr`)

Inspect multi-drug resistance risk factors and prepare them for merges without further transformation.

In [8]:
tbl_factores_riesgo_bmr = dataframes['tbl_factores_riesgo_bmr']
print(tbl_factores_riesgo_bmr)

      person_id fecha_ingreso_urgencias  hospit_ano_previo  hospit_mes_previo  \
0             1              2021-04-22                  1                  1   
1             2              2021-05-07                  1                  1   
2             3              2021-06-04                  1                  1   
3             4              2021-06-22                  1                  1   
4             5              2021-03-24                  0                  0   
...         ...                     ...                ...                ...   
3908       3909              2023-01-18                  0                  0   
3909       3910              2023-04-23                  0                  0   
3910       3911              2023-04-10                  1                  1   
3911       3912              2023-06-19                  0                  0   
3912       3913              2023-06-26                  0                  0   

      hospit_ano_previo_uci

## Symptom presentation (`tbl_sintomas`)

- Pivot symptom duration into wide format to accumulate days per symptom.
- Derive presence/absence binaries and categorical duration buckets (0: none, 1: acute ≤7d, 2: prolonged >7d).

In [9]:
sintom_map = tbl_codes2names[tbl_codes2names["variable"] == "sintoma"][["value", "name"]].to_dict(orient="records")
sintom_map = {str(float(x["value"])): x["name"].split(" | ")[-1] for x in sintom_map}
sintom_map

{'1.0': 'fiebre',
 '2.0': 'tos',
 '3.0': 'dificultad para respirar',
 '4.0': 'dolor costal',
 '5.0': 'disuria',
 '6.0': 'síndrome de disuria - polaquiuria',
 '7.0': 'tenesmo de ano y/o recto',
 '8.0': 'dolor en el ángulo renal',
 '9.0': 'náuseas',
 '10.0': 'vómitos',
 '11.0': 'dolor abdominal',
 '12.0': 'diarrea',
 '13.0': 'lesión de la piel',
 '14.0': 'lesión de mucosa',
 '15.0': 'cefalea',
 '16.0': 'dolor articular'}

### Pivot and encode symptom table

Map coded symptoms to names, then create two representations: binary presence/absence and ordinal duration buckets (none / acute / prolonged).

In [10]:
print("------ Table original ------")
print(dataframes['tbl_sintomas'].head())
tbl_sintomas = dataframes['tbl_sintomas'].copy()
tbl_sintomas["sintoma"] = tbl_sintomas["sintoma"].astype(str)
tbl_sintomas["sintoma"] = tbl_sintomas["sintoma"].map(sintom_map)
tbl_sintomas["sintoma"] = "sintoma_" + tbl_sintomas["sintoma"]

print("------ Table decoded ------")
print(tbl_sintomas[tbl_sintomas["person_id"] == 4].head())

# Check for duplicates in the original table based on person_id, fecha_ingreso_urgencias, and sintoma
# If found, prnt them and drop them before pivoting the table
print("------ Checking for duplicates ------")
num_dupes = tbl_sintomas.duplicated(
    subset=["person_id", "fecha_ingreso_urgencias", "sintoma"]
).sum()

dupes = tbl_sintomas[
    tbl_sintomas.duplicated(
        subset=["person_id", "fecha_ingreso_urgencias", "sintoma"],
        keep=False
    )
].sort_values(["person_id", "fecha_ingreso_urgencias", "sintoma"])

if num_dupes > 0:
    print(f"WARNING: {num_dupes} duplicates found. Dropping them.")
    print(f"Duplicate rows:")
    print(dupes)
    tbl_sintomas = tbl_sintomas.drop_duplicates(
        subset=["person_id", "fecha_ingreso_urgencias", "sintoma"])
else:
    print("No duplicates found.")

# Pivot dedupled table
tbl_sintomas_pivoted = tbl_sintomas.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],  
    columns="sintoma",
    values="duracion_sintoma",
    fill_value=0).reset_index()

print("------ Table pivoted ------")
print(tbl_sintomas_pivoted[tbl_sintomas_pivoted["person_id"] == 4])

# One-hot encode the presence or absence of symptoms based on the duration of symptoms
presence_absence= tbl_sintomas_pivoted.copy()
# Convert duration of symptoms to presence (1) or absence (0)
presence_absence.iloc[:, 2:] = (presence_absence.iloc[:, 2:] > 0).astype(int)

recategorized= tbl_sintomas_pivoted.copy()
# Recategorize the duration of symptoms into 3 categories: 0 (no symptom), 1 (symptom duration <= 7 days), 2 (symptom duration > 7 days)
recategorized.iloc[:, 2:] = recategorized.iloc[:, 2:].applymap(
    lambda x:0 if x== 0 else (1 if x <= 7 else 2)
)

tbl_sintomas_complete = presence_absence.merge(
    recategorized, on= ['person_id', 'fecha_ingreso_urgencias'], suffixes= ("", "_categorico")
)

print ("------ Table complete ------")
print(tbl_sintomas_complete)

------ Table original ------
   person_id fecha_ingreso_urgencias  sintoma  duracion_sintoma
0          1              2021-04-22      1.0               2.0
1          2              2021-05-07      1.0               2.0
2          3              2021-06-04      1.0               2.0
3          4              2021-06-22      1.0               1.0
4          5              2021-03-24      1.0               1.0
------ Table decoded ------
   person_id fecha_ingreso_urgencias         sintoma  duracion_sintoma
3          4              2021-06-22  sintoma_fiebre               1.0
------ Checking for duplicates ------
Duplicate rows:
      person_id fecha_ingreso_urgencias          sintoma  duracion_sintoma
3476       1690              2023-10-16  sintoma_vómitos               1.0
3478       1690              2023-10-16  sintoma_vómitos               1.0
6694       3127              2023-04-25  sintoma_náuseas               6.0
6695       3127              2023-04-25  sintoma_náuseas       

/tmp/ipykernel_1881424/4190329521.py:51: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  recategorized.iloc[:, 2:] = recategorized.iloc[:, 2:].applymap(


## Vital sign measurements (`tbl_signos`)

Parse numeric readings, normalise units, and derive clinical flags (hypotension, tachypnea, tachycardia, hypoxemia).

In [11]:
dataframes["tbl_signos"].columns[2:]

Index(['temperatura', 'hipotermia_hipertermia', 'frec_cardiaca', 'taquicardia',
       'frec_respiratoria', 'taquipnea', 'tension_arterial', 'hipotension',
       'saturacion_o2', 'hipoxemia'],
      dtype='object')

In [12]:
tbl_signos = dataframes['tbl_signos']

# for each signo extract the numeric value using regex and convert to float
for col in tbl_signos.columns[2:]:
    tbl_signos[col] = tbl_signos[col].apply(lambda x: re.findall(r'\d+\.\d+|\d+', str(x)))
    tbl_signos[col] = tbl_signos[col].apply(lambda x: float(x[0]) if x else None)

# hipotermia_hipertermia, hipotension, taquipnea, taquicardia, hipoxemia are already present in the dataset, but recatogarized? Maybe new thresholds?
# categorize temperatura into 0 if >= 36 and < 38, 1 if < 36, 2 if >= 38
tbl_signos['hipotermia_hipertermia'] = np.where(tbl_signos['temperatura'] >= 38, 2,
                                np.where(tbl_signos['temperatura'] >= 36, 0, 1))
tbl_signos['hipotermia_hipertermia'] = tbl_signos['hipotermia_hipertermia'].where(tbl_signos['temperatura'].notna())

#tbl_signos = tbl_signos.drop(columns=['hipotermia_hipertermia'])
tbl_signos['hipotension'] = np.where(tbl_signos['tension_arterial'].isna(), tbl_signos['hipotension'],
                                     np.where(tbl_signos['tension_arterial'] <= 100, 1, 0))
tbl_signos['taquipnea'] = np.where(tbl_signos['frec_respiratoria'].isna(), tbl_signos['taquipnea'],
                                   np.where(tbl_signos['frec_respiratoria'] > 20, 1, 0))
tbl_signos['taquicardia'] = np.where(tbl_signos['frec_cardiaca'].isna(), tbl_signos['taquicardia'],
                                     np.where(tbl_signos['frec_cardiaca'] > 90, 1, 0))
tbl_signos['hipoxemia'] = np.where(tbl_signos['saturacion_o2'].isna(), tbl_signos['hipoxemia'],
                                     np.where(tbl_signos['saturacion_o2'] > 90, 0, 1))
print(tbl_signos)

      person_id fecha_ingreso_urgencias  temperatura  hipotermia_hipertermia  \
0             1              2021-04-22         37.2                       0   
1             2              2021-05-07         37.0                       0   
2             3              2021-06-04         37.9                       0   
3             4              2021-06-22         37.8                       0   
4             5              2021-03-24         39.0                       2   
...         ...                     ...          ...                     ...   
3908       3909              2023-01-18         38.0                       2   
3909       3910              2023-04-23         37.6                       0   
3910       3911              2023-04-10         37.8                       0   
3911       3912              2023-06-19         38.0                       2   
3912       3913              2023-06-26         37.6                       0   

      frec_cardiaca  taquicardia  frec_

## Apply IQR-based cleaning to vital sign variables

Use the helper to remove extreme values from key vital sign metrics before merging them into the main dataset.

In [13]:
variables= ['temperatura', 'frec_respiratoria', 'frec_cardiaca', 'tension_arterial','saturacion_o2']
tbl_signos = remove_outliers_and_categorise(tbl_signos, variables)

Variable 'temperatura' recoded in the column 'temperatura_recoded'.
9 outliers replaced with NaN in the variable 'frec_respiratoria'.
293     46.0
553     50.0
2325    48.0
2400    50.0
2442    52.0
2481    52.0
3433    47.0
3461    51.0
3477    52.0
Name: frec_respiratoria, dtype: float64
Variable 'frec_respiratoria' recoded in the column 'frec_respiratoria_recoded'.
Variable 'frec_cardiaca' recoded in the column 'frec_cardiaca_recoded'.
Variable 'tension_arterial' recoded in the column 'tension_arterial_recoded'.
70 outliers replaced with NaN in the variable 'saturacion_o2'.
385     75.0
401     80.0
617     78.0
620     75.0
638     80.0
        ... 
3652    70.0
3660    70.0
3670    75.0
3810    70.0
3818    70.0
Name: saturacion_o2, Length: 70, dtype: float64
Variable 'saturacion_o2' recoded in the column 'saturacion_o2_recoded'.


## Sepsis assessment (`tbl_sepsis`)

Numerically encode sepsis questionnaire responses (e.g., lactate) and run the outlier-handling helper on selected labs.

In [14]:
tbl_sepsis = dataframes['tbl_sepsis']
# recode lactato_serico into 0 if <= 2 millimole per liter, 1 if > 2 millimole per liter
tbl_sepsis['lactato_serico'] = np.where(tbl_sepsis['lactato_serico'] == '<= 2 millimole per liter', 0, 1)
for col in tbl_sepsis.columns[2:]:
    tbl_sepsis[col] = tbl_sepsis[col].apply(lambda x: re.findall(r'\d+\.\d+|\d+', str(x)))
    tbl_sepsis[col] = tbl_sepsis[col].apply(lambda x: float(x[0]) if len(x) > 0 else None)
print(tbl_sepsis)

# Remove outliers and recode
variables =['proteina_c_reactiva']
tbl_sepsis = remove_outliers_and_categorise(tbl_sepsis, variables)

## foco map
######

      person_id fecha_ingreso_urgencias  foco  sepsis  shock_septico  sofa  \
0             1              2021-04-22   1.0     1.0            0.0   6.0   
1             2              2021-05-07   8.0     0.0            0.0   4.0   
2             3              2021-06-04  12.0     0.0            0.0   5.0   
3             4              2021-06-22  12.0     1.0            0.0   6.0   
4             5              2021-03-24   1.0     0.0            0.0   4.0   
...         ...                     ...   ...     ...            ...   ...   
3908       3909              2023-01-18   4.0     0.0            0.0   1.0   
3909       3910              2023-04-23   3.0     1.0            0.0   2.0   
3910       3911              2023-04-10   4.0     1.0            0.0   3.0   
3911       3912              2023-06-19   2.0     1.0            0.0   2.0   
3912       3913              2023-06-26   4.0     0.0            0.0   1.0   

      respiracion  snc_glasgow  cardiovascular  bilirrubina  pl

## Previous infection history (`tbl_infecciones_previas`)

- Map microorganism SNOMED codes to clinical categories and pivot them into dummy features.
- Count infection events per patient and compute inter-event timing statistics.

In [15]:
# Load table with classification group for each organism in column label
organism_classification = dataframes["tbl_microorganismos"].copy()

# Rename default values for final classification, use _ to separate target variables
label_map = {
    "NOEB": "_Other bacteria",
    "VIRUS": "_Virus",
    "FUNGUS": "_Fungi",
    "OEB": "_Enterobacteria",
    "ECOLI": "Escherichia coli",
    "SA": "Staphylococcus aureus",
    "PSA": "Pseudomonas aeruginosa",
    "KP": "Klebsiella pneumoniae",
    "SP": "Streptococcus pneumoniae",
    "EC": "Enterococcus"
}
organism_classification["label"] = organism_classification["label"].map(label_map)
# organism_classification[organism_classification.duplicated(subset="snomed_code", keep=False)].sort_values(by=["snomed_code", "label"])
# There are snomed_codes with 2 labels, so always keep the most informative
organism_classification = organism_classification.sort_values(by=["snomed_code", "label"]).drop_duplicates(subset="snomed_code", keep="first")
# Create a dictionary with {organism-snomed-code (value) : classification group (label)}
organism_codes_map = dict(zip(organism_classification["snomed_code"], organism_classification["label"])) #label
print(organism_codes_map)

{'10021006': '_Enterobacteria', '10049004': '_Other bacteria', '10091000087105': '_Other bacteria', '10151000087104': '_Fungi', '1017006': '_Other bacteria', '10171000087105': '_Fungi', '10210008': '_Fungi', '10262005': 'Enterococcus', '10334001': '_Enterobacteria', '103427005': '_Other bacteria', '103428000': '_Other bacteria', '103429008': 'Escherichia coli', '103434007': '_Enterobacteria', '103435008': '_Enterobacteria', '103436009': 'Enterococcus', '103437000': 'Enterococcus', '103438005': 'Enterococcus', '103447002': '_Other bacteria', '103448007': '_Other bacteria', '103449004': '_Other bacteria', '103450004': '_Other bacteria', '103452007': '_Other bacteria', '103453002': '_Other bacteria', '103454008': '_Other bacteria', '103455009': '_Other bacteria', '103459003': '_Other bacteria', '103460008': '_Other bacteria', '103461007': '_Other bacteria', '103462000': '_Other bacteria', '103474001': '_Other bacteria', '103475000': '_Other bacteria', '103476004': '_Other bacteria', '1034

### Pivot organism groups into binary feature columns

For each previous infection, assign its microorganism to a clinical group and pivot into wide format so each group becomes a binary feature column.

In [16]:
tbl_infecciones_previas_mer = dataframes['tbl_infecciones_previas'].copy()

# Pasamos a dummy los microorganismos y rellenamos la información con el sumatorio de cada dummy. De este modo no perdemos información.
# Finalmente lo almacenamos en tbl_infecciones_previas_microorganismo
tbl_infecciones_previas_mer['grupo_microorganismo'] = tbl_infecciones_previas_mer['microorganism_infec_prev'].astype(str).map(organism_codes_map)
tbl_infecciones_previas_mer = tbl_infecciones_previas_mer.drop(columns=['microorganism_infec_prev', 'bmr_infec_previa', 'feno_resist_infec_prev'])
tbl_infecciones_previas_mer["dummy"] = 1
list_organisms = tbl_infecciones_previas_mer["grupo_microorganismo"].dropna().unique()
#tbl_infecciones_previas_mer = pd.get_dummies(tbl_infecciones_previas_mer, columns=['feno_resist_infec_prev'])
tbl_infecciones_previas_microorganismos = tbl_infecciones_previas_mer.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],  
    columns="grupo_microorganismo",
    values="dummy",
    aggfunc="sum",
    fill_value=0).reset_index()

# Nos quedamos solo con las columnas binarias, > 1 se considera que el paciente ha tenido esa infección previa, independientemente de las veces que la haya tenido. 
# Por lo tanto, recodificamos a 1 si el valor es >= 1, y a 0 si es 0. Finalmente, eliminamos las columnas originales con el número de infecciones previas.
for col in list_organisms:
    bin_col = col+"_binary"
    tbl_infecciones_previas_microorganismos[bin_col] = np.where(
        tbl_infecciones_previas_microorganismos[col] >= 1, 1, 0
    )
    tbl_infecciones_previas_microorganismos = tbl_infecciones_previas_microorganismos.drop(columns=col)

### Inspect unique organism groups

Exploratory cell to verify which organism groups exist in the previous infection table.

In [17]:
list_organisms

array(['Enterococcus', '_Other bacteria', '_Fungi', '_Virus',
       'Escherichia coli', 'Klebsiella pneumoniae',
       'Streptococcus pneumoniae', '_Enterobacteria',
       'Pseudomonas aeruginosa', 'Staphylococcus aureus'], dtype=object)

## Antibiotic treatments prior to admission (`tbl_tratamiento_antibiotico_previo`)

Filter to patients receiving pre-admission antimicrobials, aggregate treatment durations, and derive summary indicators.

In [18]:
tbl_antib_prev = copy.deepcopy(dataframes["tbl_tratamiento_antibiotico_previo"])
tbl_antib_prev["antib_previo_si_no"] = np.where(tbl_antib_prev["dias_trat_antimicrobiano"] > 0, 1, 0)
tbl_antib_prev = tbl_antib_prev[tbl_antib_prev["antib_previo_si_no"] == 1]

### Map antibiotic codes to names in ultimo_antib column
antibmap = tbl_codes2names[tbl_codes2names["variable"] == "antimicrobiano_previo"][["value", "name"]].to_dict(orient="records")
antibmap = {x["value"]: x["name"].split(" | ")[-1].split("; ")[0] for x in antibmap}

tbl_antib_prev["antimicrobiano_previo"] = tbl_antib_prev["antimicrobiano_previo"].map(antibmap)
tbl_antib_prev["prev_betalactamase_inhib"] = (
    tbl_antib_prev["antimicrobiano_previo"]
    .str.contains("beta-lactamase inhibitor")
    .groupby(tbl_antib_prev["person_id"])   # adjust column name
    .transform("max")
    .astype(int)
)

### Derive 90-day pre-admission antibiotic window flag

For each antibiotic course, identify those given maximum 90 days before admission and store the drug name; older treatments are labelled NEGATIVE.

In [19]:
prev_90_mask = (
    pd.to_datetime(tbl_antib_prev["fecha_ingreso_urgencias"], errors="coerce") 
    - pd.to_datetime(tbl_antib_prev["fecha_administracion_antib"], errors="coerce") 
    < pd.Timedelta(days=90)
)
tbl_antib_prev["antimicrobiano_previo"] = (
    tbl_antib_prev["antimicrobiano_previo"].where(prev_90_mask)
).fillna("NEGATIVE")


### Group antimicrobial treatment based on clinical criteria

In [20]:
antimicrobial_groups = [
    ("AMIKACINA", "Aminoglucósidos", "amikacin"),
    ("AMOXICILINA", "Penicilinas", "amoxicillin"),
    ("AMOXICILINA / CLAVULANICO", "Penicilinas", "amoxicillin and beta-lactamase inhibitor"),
    ("AMPICILINA", "Penicilinas", "ampicillin"),
    ("AZITROMICINA", "Macrólidos", "azithromycin"),
    ("AZTREONAM", "Monobactámicos", "aztreonam"),
    ("BENCILPENICILINA", "Penicilinas", None),
    ("BENCILPENICILINA-BENZATINA", "Penicilinas", None),
    ("CEFADROXILO MONOHIDRATO", "Cefalosporinas 1 gen", "cefadroxil"),
    ("CEFAZOLINA", "Cefalosporinas 1 gen", "cefazolin"),
    ("CEFEPIMA", "Cefalosporinas 4 gen", "cefepime"),
    ("CEFIDEROCOL", "Cefalosporinas 4 gen", None),
    ("CEFIXIMA", "Cefalosporinas 3 gen", "cefixime"),
    ("CEFOTAXIMA", "Cefalosporinas 3 gen", "cefotaxime"),
    ("CEFTAROLINA FOSAMILO", "Cefalosporinas 2 gen", "ceftaroline fosamil"),
    ("CEFTAZIDIMA", "Cefalosporinas 3 gen", "ceftazidime"),
    ("CEFTAZIDIMA / AVIBACTAM", "Cefalosporinas 3 gen", "ceftazidime and beta-lactamase inhibitor"),
    ("CEFTOLOZANO / TAZOBACTAM", "Cefalosporinas 3 gen", "ceftolozane and beta-lactamase inhibitor"),
    ("CEFTRIAXONA", "Cefalosporinas 3 gen", "ceftriaxone"),
    ("CEFUROXIMA", "Cefalosporinas 2 gen", "cefuroxime"),
    ("CIPROFLOXACINO", "Quinolonas", "ciprofloxacin"),
    ("CLARITROMICINA", "Macrólidos", "clarithromycin"),
    ("CLINDAMICINA", "Lincosamidas", "clindamycin"),
    ("CLOXACILINA", "Penicilinas", "cloxacillin"),
    ("COLISTIMETATO DE SODIO", "Polimixinas", None),
    ("DALBAVANCINA", "Glicopéptidos", "dalbavancin"),
    ("DAPTOMICINA", "Lipopéptidos", "daptomycin"),
    ("DOXICICLINA", "Tetraciclinas", "doxycycline"),
    ("ERITROMICINA", "Macrólidos", "erythromycin"),
    ("ERTAPENEM", "Carbapenemas", "ertapenem"),
    ("FIDAXOMICINA", "Macrólidos", None),
    ("FOSFOMICINA", "Fosfomicina", "fosfomycin"),
    ("FOSFOMICINA-TROMETAMOL", "Fosfomicina", "fosfomycin"),
    ("GENTAMICINA", "Aminoglucósidos", "gentamicin"),
    ("IMIPENEM", "Carbapenemas", "imipenem and cilastatin"),
    ("IMIPENEM/RELEBACTAM", "Carbapenemas", None),
    ("LEVOFLOXACINO", "Quinolonas", "levofloxacin"),
    ("LINEZOLID", "Lincosamidas", "linezolid"),
    ("MEROPENEM", "Carbapenemas", "meropenem"),
    ("METRONIDAZOL", "Metronidazol", "metronidazole"),
    ("MINOCICLINA", "Minociclina", None),
    ("MOXIFLOXACINO", "Quinolonas", "moxifloxacin"),
    ("NITROFURANTOINA", "Nitrofurantoína", "nitrofurantoin"),
    ("NORFLOXACINO", "Quinolonas", "norfloxacin"),
    ("PIPERACILINA / TAZOBACTAM", "Penicilinas", "piperacillin and beta-lactamase inhibitor"),
    ("POSACONAZOL", "Azoles", "posaconazole"),
    ("SULFADIAZINA", "Sulfonamidas", "sulfadiazine"),
    ("SULFAMETOXAZOL / TRIMETOPRIMA", "Sulfonamidas", "sulfamethoxazole and trimethoprim"),
    ("TRIMETOPRIMA", "Sulfonamidas", "trimethoprim"),
    ("TEICOPLANINA", "Glicopéptidos", "teicoplanin"),
    ("TIGECICLINA", "Tigeciclina", "tigecycline"),
    ("TOBRAMICINA", "Aminoglucósidos", "tobramycin"),
    ("VANCOMICINA", "Glicopéptidos", "vancomycin"),
    ("A", "Tetraciclinas", "tetracycline"),
    ("A", "Cefalosporinas 3 gen", "cefditoren"),
    ("A", "Carbapenemas", "meropenem and vaborbactam"),
    ("A", "Cefalosporinas 1 gen", "cefalexin"),
    ("A", "Cefalosporinas 2 gen", "cefoxitin"),
    ("A", "AntiTuberculoso", "rifampicin"),
    ("A", "AntiTuberculoso","rifabutin"),
    ("A", "AntiTuberculoso","isoniazid"),
    ("A", "AntiTuberculoso","pyrazinamide"),
    ("A", "AntiTuberculoso","ethambutol"),
    ("B", "Azoles", "isavuconazole"),
    ("B", "Azoles", "fluconazole"),
    ("B", "Azoles", "itraconazole"),
    ("B", "Azoles", "voriconazole"),
    ("B", "Equinocandinas", "caspofungin"),
    ("B", "Equinocandinas", "micafungin"),
    ("B", "Equinocandinas", "anidulafungin"),
]
# transform english code into spanish antib group
antimicrobial_groups = {x[2]: x[1] for x in antimicrobial_groups}
tbl_antib_prev["antimicrobiano_previo"] = tbl_antib_prev["antimicrobiano_previo"].map(antimicrobial_groups)
tbl_antib_prev["antimicrobiano_previo"].value_counts()

antimicrobiano_previo
Penicilinas             779
Quinolonas              503
Cefalosporinas 3 gen    416
Carbapenemas            319
Cefalosporinas 2 gen    152
Sulfonamidas            151
Glicopéptidos           139
Fosfomicina             137
Macrólidos              120
Lincosamidas             76
Lipopéptidos             53
Azoles                   45
Aminoglucósidos          40
Cefalosporinas 1 gen     28
Cefalosporinas 4 gen     20
AntiTuberculoso          16
Metronidazol             15
Nitrofurantoína          12
Tetraciclinas             9
Equinocandinas            7
Monobactámicos            6
Tigeciclina               5
Name: count, dtype: int64

### Date-clean and compute days since last antibiotic

Remove future-dated rows, find the last antibiotic event per patient, and compute elapsed days to admission.

In [21]:
# Parse date columns to datetime for proper arithmetic
tbl_antib_prev["fecha_ingreso_urgencias"] = pd.to_datetime(tbl_antib_prev["fecha_ingreso_urgencias"], errors="coerce")
tbl_antib_prev["fecha_administracion_antib"] = pd.to_datetime(tbl_antib_prev["fecha_administracion_antib"], errors="coerce")

# Remove rows where the antibiotic date is AFTER the admission date (data entry errors)
tbl_antib_prev = tbl_antib_prev[tbl_antib_prev["fecha_administracion_antib"] <= tbl_antib_prev["fecha_ingreso_urgencias"]]

# Find the row with the latest admission date per patient — represents the most recent treatment episode
idx = tbl_antib_prev.groupby("person_id")["fecha_ingreso_urgencias"].idxmax()
last = tbl_antib_prev.loc[idx, ["person_id", "antimicrobiano_previo", "fecha_ingreso_urgencias", "fecha_administracion_antib"]].copy()

# Compute days elapsed between most recent antibiotic administration and ER admission
last["dias_ultimo_antib"] = (
    (last["fecha_ingreso_urgencias"] - last["fecha_administracion_antib"]).dt.total_seconds() / 86400.0
)

# Broadcast the last-antibiotic info back to all rows of the same patient
tbl_antib_prev["ultimo_antib"] = last.set_index("person_id")["antimicrobiano_previo"].reindex(tbl_antib_prev["person_id"]).to_numpy()
tbl_antib_prev["dias_ultimo_antib"] = last.set_index("person_id")["dias_ultimo_antib"].reindex(tbl_antib_prev["person_id"]).to_numpy()

# Convert back to string for consistent dtype across joins
tbl_antib_prev["fecha_ingreso_urgencias"] = tbl_antib_prev["fecha_ingreso_urgencias"].astype(str)
print(tbl_antib_prev)

      person_id fecha_ingreso_urgencias fecha_administracion_antib  \
1             2              2021-05-07                 2021-04-22   
2             3              2021-06-04                 2021-05-12   
3             3              2021-06-04                 2021-05-09   
4             3              2021-06-04                 2021-05-08   
5             3              2021-06-04                 2021-04-22   
...         ...                     ...                        ...   
5882       3882              2023-05-05                 2023-02-06   
5887       3886              2023-01-30                 2023-01-13   
5900       3899              2023-06-08                 2023-06-02   
5909       3908              2023-01-01                 2022-12-07   
5912       3911              2023-04-10                 2023-03-15   

     antimicrobiano_previo via_administ_antib_prev  dias_trat_antimicrobiano  \
1             Carbapenemas                    None                       5.0   

### Pivot prior antibiotic use to wide format

Create one binary column per pharmacological family indicating whether the patient received at least one course of that family before admission.

In [22]:
"""
# Code to include dias_trat_antimicrobiano column
antib_prev_pivoted = tbl_antib_prev.drop(columns="via_administ_antib_prev").pivot_table(
    index=["person_id", "fecha_ingreso_urgencias", "antib_previo_si_no", "ultimo_antib", "dias_ultimo_antib", "prev_betalactamase_inhib"],
    columns="antimicrobiano_previo",
    values="dias_trat_antimicrobiano",
    aggfunc="max",
    fill_value=0
).reset_index()"""

# Collect all pharmacological families present in the data (used to create binary columns)
list_antibs = tbl_antib_prev["antimicrobiano_previo"].dropna().unique().tolist()

# Pivot to presence/absence (1 = patient had at least one treatment in this drug family)
antib_prev_pivoted = (
    tbl_antib_prev
    .drop(columns="via_administ_antib_prev")
    .assign(presence=1)          # sentinel value for counting events
    .pivot_table(
        index=[
            "person_id",
            "fecha_ingreso_urgencias",
            "antib_previo_si_no",
            "ultimo_antib",
            "dias_ultimo_antib",
            "prev_betalactamase_inhib",
        ],
        columns="antimicrobiano_previo",
        values="presence",
        aggfunc="sum",           # sum of sentinels = event count per family
        fill_value=0,
    )
    .reset_index()
)

# Convert event counts → binary flags and drop raw count columns
# drop columns with total count per antib prev
for column in list_antibs:
    bin_colname = column + "_binary"
    antib_prev_pivoted[bin_colname] = np.where(antib_prev_pivoted[column] >= 1, 1, 0)
    antib_prev_pivoted = antib_prev_pivoted.drop(columns=column)

# Total number of distinct drug families the patient was exposed to
antib_prev_pivoted["antib_previo_total_veces"] = antib_prev_pivoted.iloc[:, 5:].sum(axis=1)

# Working copy used during final merge step
antib_prev_to_merge = antib_prev_pivoted.copy()

### Inspect pivoted antibiotic table

Exploratory display of the final wide-format prior-antibiotic feature table.

In [23]:
# Exploratory: display pivoted prior antibiotic table
antib_prev_pivoted

antimicrobiano_previo,person_id,fecha_ingreso_urgencias,antib_previo_si_no,ultimo_antib,dias_ultimo_antib,prev_betalactamase_inhib,Carbapenemas_binary,Lincosamidas_binary,Glicopéptidos_binary,Quinolonas_binary,...,Tigeciclina_binary,Monobactámicos_binary,Tetraciclinas_binary,Fosfomicina_binary,Cefalosporinas 1 gen_binary,Azoles_binary,Nitrofurantoína_binary,Equinocandinas_binary,Cefalosporinas 4 gen_binary,antib_previo_total_veces
0,2,2021-05-07,1,Carbapenemas,15.0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,3,2021-06-04,1,Lincosamidas,23.0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,3
2,4,2021-06-22,1,Quinolonas,14.0,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,4
3,7,2020-05-15,1,Glicopéptidos,15.0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
4,15,2021-01-23,1,Quinolonas,23.0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1336,3882,2023-05-05,1,Sulfonamidas,88.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1337,3886,2023-01-30,1,Cefalosporinas 4 gen,17.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1338,3899,2023-06-08,1,Quinolonas,6.0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
1339,3908,2023-01-01,1,Penicilinas,25.0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2


## Hemoculture results from the emergency department

Deep-copy the hemoculture table, map microorganism codes to organism groups, and inject clinician-reviewed co-infection resolutions for the target label.

In [24]:
import copy

# Deep-copy to avoid mutating the original database table
hemo_urg = copy.deepcopy(dataframes['tbl_hemocultivo_de_urgencias'])

# Map microorganism SNOMED codes to clinical group labels.
# NaN codes are treated as 0 (absent) and then resolved via organism_codes_map.
# Codes not found in the map default to 'NEGATIVE'.
hemo_urg["microorganismo"] = (
    hemo_urg["microorganismo"]
    .fillna("0")
    .astype(int)
    .astype(str)
    .map(lambda x: organism_codes_map.get(x, "NEGATIVE"))
)
hemo_urg["microorganismo"].value_counts()

microorganismo
NEGATIVE                    2074
Escherichia coli            1420
Klebsiella pneumoniae        504
_Enterobacteria              293
_Other bacteria              273
Staphylococcus aureus        199
Pseudomonas aeruginosa       111
Streptococcus pneumoniae     106
Enterococcus                  56
_Fungi                         8
Name: count, dtype: int64

In [25]:
print(hemo_urg.head())

   person_id fecha_ingreso_urgencias id_hemocultivo fecha_hemocultivo  \
0          1              2021-04-22          621-2        2021-04-22   
1          2              2021-05-07      212164727        2021-05-07   
2          3              2021-06-04          621-4        2021-06-04   
3          4              2021-06-22          621-5        2021-06-22   
4          5              2021-03-24         621-10        2021-03-24   

   hemo_positivo_si_no microorganismo  bmr_etiologia  fenotipo_resistencia  
0                  0.0       NEGATIVE            NaN                   NaN  
1                  1.0   Enterococcus            0.0                   NaN  
2                  0.0       NEGATIVE            NaN                   NaN  
3                  0.0       NEGATIVE            NaN                   NaN  
4                  0.0       NEGATIVE            NaN                   NaN  


In [29]:
conflicts = (
    hemo_urg
    .groupby(["person_id", "id_hemocultivo"])["bmr_etiologia"]
    .nunique()
    .reset_index()
    .query("bmr_etiologia > 1")
)
conflicts

,person_id,id_hemocultivo,bmr_etiologia
132,133,212285082,2
461,462,12651437,2
772,770,40453674,2
873,871,40427505,2
963,961,15490755,2
1091,1089,05035094,2
1112,1110,15470942,2
1229,1228,05031009,2
1242,1241,05032380,2
1312,1309,5053832,2


### Clinician-reviewed co-infection resolution

For patients with multiple organisms detected, overwrite the automatically mapped label with the clinician-determined dominant organism.

In [31]:
# Manual clinician-reviewed co-infection resolutions.
# For patients with multiple organisms in their blood culture (co-infections),
# the dominant / clinically significant organism was determined manually and stored here.
# Keys are person_id integers; values are the agreed organism name.
coinf_res_dict = {
    50: "Escherichia coli",
    419: "Staphylococcus aureus",
    770: "Escherichia coli",
    780: "Escherichia coli",
    1214: "Klebsiella pneumoniae",
    1309: "Escherichia coli",
    1886: "Klebsiella pneumoniae",
    1925: "Escherichia coli",
    1960: "Enterococcus avium",
    1977: "Escherichia coli",
    2274: "Escherichia coli",
    2294: "Contaminación",
    2634: "Escherichia coli",
    2638: "Klebsiella pneumoniae",
    2681: "Klebsiella pneumoniae",
    2845: "Escherichia coli",
    2971: "Escherichia coli",
    2983: "Escherichia coli",
    3019: "Escherichia coli",
    3087: "Escherichia coli",
    3167: "Escherichia coli",
    3352: "Escherichia coli",
    3479: "Escherichia coli",
    3819: "Otra enterobacteria",
    1247: "Otra enterobacteria",
    2640: "Escherichia coli",
    133: "Pseudomonas aeruginosa",
    1881: "Escherichia coli",
    420: "Pseudomonas fluorescens",
    1110: "Clostridium perfringens",
    1239: "Klebsiella pneumoniae",
    1310: "Escherichia coli",
    1379: "Pseudomonas aeruginosa",
    1906: "Candida glabrata",
    2041: "Escherichia coli",
    2284: "Fusobacterium nucleatum",
    2647: "Otra enterobacteria",
    2848: "Escherichia coli",
    3116: "Escherichia coli",
    447: "Escherichia coli",
    607: "Otra enterobacteria",
    668: "Klebsiella pneumoniae",
    760: "Escherichia coli",
    956: "Streptococcus anginosus",
    1089: "Staphylococcus aureus",
    1170: "Escherichia coli",
    1228: "Escherichia coli",
    1286: "Escherichia coli",
    1308: "Klebsiella pneumoniae",
    1311: "Candida glabrata",
    1322: "Otra enterobacteria",
    1355: "Escherichia coli",
    1663: "Escherichia coli",
    1717: "Contaminación",
    1865: "Bacteroides fragilis",
    1885: "Otra enterobacteria",
    1893: "Pseudomonas aeruginosa",
    1967: "Escherichia coli",
    1975: "Escherichia coli",
    1976: "Otra enterobacteria",
    2336: "Escherichia coli",
    2441: "Escherichia coli",
    2469: "Escherichia coli",
    2926: "Klebsiella pneumoniae",
    2974: "Klebsiella pneumoniae",
    3457: "Otra enterobacteria",
    1954: "Escherichia coli",
    2016: "Escherichia coli",
    2810: "Escherichia coli",
    2840: "Staphylococcus aureus",
    429: "Streptococcus pyogenes",
    473: "Escherichia coli",
    480: "Staphylococcus aureus",
    766: "Contaminación",
    797: "Otra enterobacteria",
    1303: "Contaminación",
    1777: "Salmonella enterica",
    1839: "Escherichia coli",
    1850: "Escherichia coli",
    1873: "Otra enterobacteria",
    2308: "Contaminación",
    2335: "Otra enterobacteria",
    2341: "Staphylococcus aureus",
    2513: "Otra enterobacteria",
    2873: "Klebsiella pneumoniae",
    462: "Escherichia coli",
    562: "Escherichia coli",
    599: "Escherichia coli",
    759: "Otra enterobacteria",
    871: "Pseudomonas aeruginosa",
    961: "Escherichia coli",
    1052: "Otra enterobacteria",
    1241: "Escherichia coli",
    1246: "Klebsiella pneumoniae",
    1333: "Escherichia coli",
    1658: "Escherichia coli",
    1776: "Escherichia coli",
    1875: "Escherichia coli",
    1956: "Otra enterobacteria",
    1966: "Escherichia coli",
    2017: "Escherichia coli",
    2303: "Staphylococcus aureus",
    2321: "Escherichia coli",
    2354: "Klebsiella pneumoniae",
    2408: "Escherichia coli",
    2467: "Escherichia coli",
    2479: "Klebsiella pneumoniae",
    2493: "Pseudomonas aeruginosa",
    2533: "Escherichia coli",
    2625: "Escherichia coli",
    2663: "Escherichia coli",
    2715: "Escherichia coli",
    2813: "Klebsiella pneumoniae",
    3491: "Otra enterobacteria",
    3675: "Klebsiella pneumoniae",
    426: "Staphylococcus aureus"
}

# Secondary map: translate the clinician-resolved organism names back to the standard group labels
coinf_mapping = {
    "Candida glabrata": "_Fungi",
    "Clostridium perfringens": "_Other bacteria",
    "Contaminación": "NEGATIVE",          # culture contamination → treated as negative
    "Enterococcus avium": "Enterococcus",
    "Escherichia coli": "Escherichia coli",
    "Fusobacterium nucleatum": "_Other bacteria",
    "Klebsiella pneumoniae": "Klebsiella pneumoniae",
    "Otra enterobacteria": "_Enterobacteria",
    "Pseudomonas aeruginosa": "Pseudomonas aeruginosa",
    "Pseudomonas fluorescens": "_Other bacteria",
    "Salmonella enterica": "_Enterobacteria",
    "Staphylococcus aureus": "Staphylococcus aureus",
    "Streptococcus anginosus": "_Other bacteria",
    "Streptococcus pyogenes": "_Other bacteria",
    "Bacteroides fragilis": "_Other bacteria",
}

# Merge both maps: {person_id → standardised group label}
coinf_res_dict = {k: coinf_mapping[v] for k, v in coinf_res_dict.items()}

# Override the automatically mapped organism with the clinician-reviewed label where available
hemo_urg["microorganismo"] = hemo_urg["person_id"].map(coinf_res_dict).fillna(hemo_urg["microorganismo"])

In [32]:
print(hemo_urg.head())

   person_id fecha_ingreso_urgencias id_hemocultivo fecha_hemocultivo  \
0          1              2021-04-22          621-2        2021-04-22   
1          2              2021-05-07      212164727        2021-05-07   
2          3              2021-06-04          621-4        2021-06-04   
3          4              2021-06-22          621-5        2021-06-22   
4          5              2021-03-24         621-10        2021-03-24   

   hemo_positivo_si_no microorganismo  bmr_etiologia  fenotipo_resistencia  
0                  0.0       NEGATIVE            NaN                   NaN  
1                  1.0   Enterococcus            0.0                   NaN  
2                  0.0       NEGATIVE            NaN                   NaN  
3                  0.0       NEGATIVE            NaN                   NaN  
4                  0.0       NEGATIVE            NaN                   NaN  


### Pivot hemoculture findings to patient-level features

Deduplicate hemoculture records, pivot microorganism flags into a wide format, and compute per-visit outcome tuples such as `resultado_hemo`.

In [33]:
# Deduplicate by patient+organism and aggregate resistance info across repeated blood draws.
# 'fenotipo_resistencia' is stored as a tuple of all phenotypes seen for that patient+organism pair.
hemo_urg_grouped = (
    hemo_urg.groupby(["person_id", "microorganismo"], as_index=False)
      .agg({
          "fecha_ingreso_urgencias": "first",    # take first admission date
          "id_hemocultivo": "first",
          "fecha_hemocultivo": "first",
          "hemo_positivo_si_no": "first",
          "bmr_etiologia": "max",                 # 1 if ANY result was BMR
          "fenotipo_resistencia": lambda x: tuple(0.0 if pd.isna(v) else v for v in x)  # collect all phenotypes
      })
)

hemo_urg_grouped_ = (
    hemo_urg.groupby(["person_id", "fecha_ingreso_urgencias", "microorganismo"], as_index=False)
      .agg({
           # take first admission date
          "id_hemocultivo": "first",
          "fecha_hemocultivo": "first",
          "hemo_positivo_si_no": "first",
          "bmr_etiologia": "max",                 # 1 if ANY result was BMR
          "fenotipo_resistencia": lambda x: tuple(0.0 if pd.isna(v) else v for v in x)  # collect all phenotypes
      })
)
hemo_urg_grouped = hemo_urg_grouped.fillna({"bmr_etiologia": 0.0})

# Pivot to wide format: rows = (patient, admission, bmr, phenotype), columns = organism, values = positive flag
hemo_urg_pivoted = hemo_urg_grouped.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias", "bmr_etiologia", "fenotipo_resistencia"],
    columns="microorganismo",
    values="hemo_positivo_si_no",
    aggfunc="sum",
    fill_value=0).reset_index()
hemo_urg_pivoted

microorganismo,person_id,fecha_ingreso_urgencias,bmr_etiologia,fenotipo_resistencia,Enterococcus,Escherichia coli,Klebsiella pneumoniae,NEGATIVE,Pseudomonas aeruginosa,Staphylococcus aureus,Streptococcus pneumoniae,_Enterobacteria,_Fungi,_Other bacteria
0,1,2021-04-22,0.0,"(0.0,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,2021-05-07,0.0,"(0.0,)",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2021-06-04,0.0,"(0.0,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,2021-06-22,0.0,"(0.0,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,2021-03-24,0.0,"(0.0,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3908,3909,2023-01-18,1.0,"(1.0,)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3909,3910,2023-04-23,1.0,"(1.0,)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3910,3911,2023-04-10,1.0,"(1.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0)",0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3911,3912,2023-06-19,1.0,"(1.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Derive `resultado_hemo` outcome tuple

Construct a per-patient tuple of all positive organisms from the pivoted hemoculture table, which serves as the primary multi-label target.

In [34]:
def get_bacteria(row):
    """Return a comma-separated string of all organism columns that are positive (value == 1)."""
    return ", ".join([col for col in row.index if row[col] == 1])

# Create a tuple of all positive organisms per patient visit (e.g. ('Escherichia coli',) or ('NEGATIVE',))
hemo_urg_pivoted["resultado_hemo"] = (
    hemo_urg_pivoted
    .drop(columns=["person_id", "fecha_ingreso_urgencias", "bmr_etiologia", "fenotipo_resistencia"])
    .apply(get_bacteria, axis=1)
)
# Empty string (no positive organisms) → NEGATIVE; convert to tuple for downstream multi-label handling
hemo_urg_pivoted["resultado_hemo"] = (
    hemo_urg_pivoted["resultado_hemo"]
    .replace("", "NEGATIVE")
    .apply(lambda x: tuple(x.split(", ")))
)

In [35]:
print(hemo_urg_pivoted)

microorganismo  person_id fecha_ingreso_urgencias  bmr_etiologia  \
0                       1              2021-04-22            0.0   
1                       2              2021-05-07            0.0   
2                       3              2021-06-04            0.0   
3                       4              2021-06-22            0.0   
4                       5              2021-03-24            0.0   
...                   ...                     ...            ...   
3908                 3909              2023-01-18            1.0   
3909                 3910              2023-04-23            1.0   
3910                 3911              2023-04-10            1.0   
3911                 3912              2023-06-19            1.0   
3912                 3913              2023-06-26            0.0   

microorganismo                 fenotipo_resistencia  Enterococcus  \
0                                            (0.0,)           0.0   
1                                            

## Colonisation history (`tbl_colonizaciones_previas`)

Map colonising organisms to grouped labels and pivot them into patient-level indicator features.

In [36]:
# Deep-copy to avoid mutating the original database table
colo_prev = copy.deepcopy(dataframes["tbl_colonizaciones_previas"])

# Map each colonising organism's SNOMED code to its clinical group label using the same map as infections
colo_prev["microorganism_colonizador"] = colo_prev["microorganism_colonizador"].astype(str).map(organism_codes_map)

### Pivot colonisation history to binary features

Create one binary column per colonising organism group, prefixed with `colo_`, at the patient level.

In [37]:
# Sentinel column for counting colonisation events during pivot
colo_prev["dummy"] = 1

# All unique colonising organism groups (used to create binary columns)
list_colorgs = colo_prev["microorganism_colonizador"].dropna().unique()

# Pivot: rows = (patient, admission), columns = organism group, values = event count
colo_prev_pivoted = colo_prev.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],
    columns="microorganism_colonizador",
    values="dummy",
    aggfunc="sum",
    fill_value=0).reset_index()

# Convert event counts → binary flags and drop raw count columns
for col in list_colorgs:
    bin_col = col + "_binary"
    colo_prev_pivoted[bin_col] = np.where(colo_prev_pivoted[col] >= 1, 1, 0)
    colo_prev_pivoted = colo_prev_pivoted.drop(columns=col)

# Drop admission date — colonisation is patient-level, not tied to a specific visit
colo_prev_pivoted = colo_prev_pivoted.drop(columns=["fecha_ingreso_urgencias"])

# Prefix all non-index columns with 'colo_' for easy identification downstream
colo_prev_pivoted.columns = [
    "colo_" + str(col) if col not in ["person_id", "fecha_ingreso_urgencias"] else col
    for col in colo_prev_pivoted.columns
]

### Compute infection event count per patient

Count the number of distinct previous infection dates per patient, then fix a known date typo before parsing.

In [38]:
# Compute number of previous infection events per patient.
# (Spanish comment: calculate number of events per patient and average time between visits)
tbl_visitas_ip = dataframes["tbl_infecciones_previas"].copy()

# Identify and report rows with malformed date strings
fechas_invalidas = tbl_visitas_ip[~tbl_visitas_ip['fecha_infeccion'].str.match(r'\d{4}-\d{2}-\d{2}')]

# Fix a known typo: '323-05-01' should be '2023-05-01'
tbl_visitas_ip.loc[tbl_visitas_ip['fecha_infeccion'] == '323-05-01', 'fecha_infeccion'] = '2023-05-01'

# Parse infection dates
tbl_visitas_ip['fecha_infeccion'] = pd.to_datetime(tbl_visitas_ip['fecha_infeccion'])

# Retain only the columns needed for timeline calculations
tbl_visitas_ip = tbl_visitas_ip[['person_id', 'fecha_ingreso_urgencias', 'fecha_infeccion']]

# Count unique infection dates per patient (proxy for number of distinct infection episodes)
num_visitas = tbl_visitas_ip.groupby('person_id')['fecha_infeccion'].nunique().reset_index(name='num_inf_previas')
print(num_visitas)

      person_id  num_inf_previas
0             3                1
1             4                1
2             8                3
3            11                1
4            14                1
...         ...              ...
1210       3872                1
1211       3873                1
1212       3882                1
1213       3886                1
1214       3899                1

[1215 rows x 2 columns]


## Time since last infection at admission

Compute visit-level timelines: number of infections per patient, days since the last event, and average gaps for recurrent cases.

In [39]:
# (Spanish comment: compute days from last infection to ER admission)
tbl_visitas_ip['fecha_ingreso_urgencias'] = pd.to_datetime(tbl_visitas_ip['fecha_ingreso_urgencias'])

# Date of the most recent previous infection per patient
ultima_infeccion = tbl_visitas_ip.groupby('person_id')['fecha_infeccion'].max().reset_index(name='ultima_fecha')
ultima_infeccion['ultima_fecha'] = pd.to_datetime(ultima_infeccion['ultima_fecha'])

# One admission date per patient (deduplicated)
fecha_ingreso_unica = tbl_visitas_ip[['person_id', 'fecha_ingreso_urgencias']].drop_duplicates()

# Merge and compute the gap in days between last infection and ER admission
ultima_infeccion_df = ultima_infeccion.merge(fecha_ingreso_unica, on='person_id')
ultima_infeccion_df['tiempo_ultima'] = (
    ultima_infeccion_df['fecha_ingreso_urgencias'] - ultima_infeccion_df['ultima_fecha']
).dt.days

### Merge previous infection timeline into one table

Combine organism-level dummies, infection count, and days-since-last-infection into a single per-patient features table.

In [40]:
# Merge organism-level features, infection count, and time-since-last-infection into one table
# (Spanish comment: merge previous infections)
tbl_infecciones_complete = tbl_infecciones_previas_microorganismos.merge(num_visitas, on=['person_id'], how='left')
tbl_infecciones_complete = tbl_infecciones_complete.merge(ultima_infeccion_df, on=['person_id'], how='left')

# Drop the duplicate admission date column introduced by the second merge
tbl_infecciones_complete = tbl_infecciones_complete.drop(columns=['fecha_ingreso_urgencias_y'])
tbl_infecciones_complete.head(5)

,person_id,fecha_ingreso_urgencias_x,Enterococcus_binary,_Other bacteria_binary,_Fungi_binary,_Virus_binary,Escherichia coli_binary,Klebsiella pneumoniae_binary,Streptococcus pneumoniae_binary,_Enterobacteria_binary,Pseudomonas aeruginosa_binary,Staphylococcus aureus_binary,num_inf_previas,ultima_fecha,tiempo_ultima
0,3,2021-06-04,1,0,0,0,0,0,0,0,0,0,1,2021-05-08,27
1,4,2021-06-22,1,0,0,0,0,0,0,0,0,0,1,2021-05-08,45
2,8,2020-04-13,0,1,1,1,0,0,0,0,0,0,3,2019-10-24,172
3,11,2020-03-31,0,1,0,0,0,0,0,0,0,0,1,2019-11-05,147
4,14,2021-02-24,0,1,0,0,1,0,0,0,0,0,1,2020-10-09,138


### Inspect available table names

Exploratory cell to verify which tables were loaded from the database.

In [41]:
# Exploratory: list all available table names in the dataframes dictionary
dataframes.keys()

dict_keys(['tbl_master', 'tbl_paciente', 'tbl_comorbilidad', 'tbl_factores_riesgo_bmr', 'tbl_sintomas', 'tbl_signos', 'tbl_sepsis', 'tbl_infecciones_previas', 'tbl_colonizaciones_previas', 'tbl_tratamiento_antibiotico_previo', 'tbl_tratamiento_empirico', 'tbl_hemocultivo_de_urgencias', 'tbl_otros_cultivos_en_urgencias', 'tbl_codes2names', 'tbl_personid2center', 'tbl_microorganismos'])

## Other emergency cultures (no resistance data)

Standardise organism codes, pivot additional culture indicators, and prepare them for merging with hemoculture outcomes.

In [42]:
tbl_otros_cultivos_en_urgencias = copy.deepcopy(dataframes['tbl_otros_cultivos_en_urgencias'])

# Map microorganism codes to group labels (same logic as hemocultures)
tbl_otros_cultivos_en_urgencias["otro_cult_microorganismo"] = (
    tbl_otros_cultivos_en_urgencias["microorganismo_otros_cult"]
    .fillna("0")
    .astype(int)
    .astype(str)
    .map(lambda x: organism_codes_map.get(x, "NEGATIVE"))
)

# Drop columns not needed for the pivot (metadata and resistance fields not available for other cultures)
tbl_otros_cultivos_en_urgencias = tbl_otros_cultivos_en_urgencias.drop(
    columns=["microorganismo_otros_cult", "id_otros_cultivos", "fecha_otros_cultivos",
             "bmr_etiologia_otros", "fenotipo_resistencia_otros"]
)

# Fill missing culture types with 0 and remove duplicates
tbl_otros_cultivos_en_urgencias["tipo_cultivo"] = tbl_otros_cultivos_en_urgencias["tipo_cultivo"].fillna(0)
tbl_otros_cultivos_en_urgencias = tbl_otros_cultivos_en_urgencias.dropna().drop_duplicates()

# Sentinel for pivot counting
tbl_otros_cultivos_en_urgencias["dummy"] = 1

# Pivot to wide format: one column per organism group
tbl_otros_cultivos_en_urgencias_pivoted = tbl_otros_cultivos_en_urgencias.pivot_table(
    index=["person_id", "fecha_ingreso_urgencias"],
    columns="otro_cult_microorganismo",
    values="dummy",
    aggfunc="sum",
    fill_value=0).reset_index()

# Prefix columns so they can be distinguished from hemoculture columns after merging
tbl_otros_cultivos_en_urgencias_pivoted = tbl_otros_cultivos_en_urgencias_pivoted.rename(
    columns=lambda x: f"otros_cult_{x}" if x not in ["person_id", "fecha_ingreso_urgencias"] else x
)
tbl_otros_cultivos_en_urgencias_pivoted

otro_cult_microorganismo,person_id,fecha_ingreso_urgencias,otros_cult_Enterococcus,otros_cult_Escherichia coli,otros_cult_Klebsiella pneumoniae,otros_cult_NEGATIVE,otros_cult_Pseudomonas aeruginosa,otros_cult_Staphylococcus aureus,otros_cult_Streptococcus pneumoniae,otros_cult__Enterobacteria,otros_cult__Fungi,otros_cult__Other bacteria,otros_cult__Virus
0,1,2021-04-22,0,0,0,1,0,0,0,0,0,0,0
1,2,2021-05-07,0,0,0,1,0,0,0,0,0,0,0
2,3,2021-06-04,0,0,0,1,0,0,0,0,0,0,0
3,4,2021-06-22,0,0,0,1,0,0,0,0,0,0,0
4,5,2021-03-24,0,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3908,3909,2023-01-18,0,1,0,0,0,0,0,0,0,0,0
3909,3910,2023-04-23,0,0,0,1,0,0,0,0,0,0,0
3910,3911,2023-04-10,0,0,1,0,0,0,0,0,0,0,0
3911,3912,2023-06-19,0,0,0,1,0,0,0,0,0,0,0


### Combine all culture sources and pick dominant organism

Merge hemocultures and other emergency cultures, then apply `pick_dominant` to select the most informative organism per patient visit.
By this, we try to capture the most hemoculture-positive events.

In [ ]:


org_list = list(set(organism_codes_map.values()))
print(org_list)

# Outer-join hemocultures and other cultures so every patient visit is represented
all_urg = tbl_otros_cultivos_en_urgencias_pivoted.merge(
    hemo_urg_pivoted, on=["person_id", "fecha_ingreso_urgencias"], how="outer"
)

print(all_urg.columns)
# Strip the 'otros_cult_' prefix so organism columns from both sources share the same name,
# then sum columns with the same name (combines counts from all culture types)
all_urg_comb = (
    all_urg.rename(columns=lambda c: c.replace('otros_cult_', ''))
      .groupby(axis=1, level=0)
      .sum()
)

# Move NEGATIVE column to the end for clarity
all_urg_comb = all_urg_comb[[c for c in all_urg_comb.columns if c != "NEGATIVE"] + ["NEGATIVE"]]

# Apply pick_dominant to select the single most informative organism per patient visit
all_urg_comb["all_cult_org"] = all_urg_comb[org_list].apply(pick_dominant, axis=1)
all_urg_comb

['_Virus', '_Fungi', 'Escherichia coli', '_Other bacteria', 'Streptococcus pneumoniae', 'Pseudomonas aeruginosa', '_Enterobacteria', 'Staphylococcus aureus', 'Enterococcus', 'Klebsiella pneumoniae']
Index(['person_id', 'fecha_ingreso_urgencias', 'otros_cult_Enterococcus',
       'otros_cult_Escherichia coli', 'otros_cult_Klebsiella pneumoniae',
       'otros_cult_NEGATIVE', 'otros_cult_Pseudomonas aeruginosa',
       'otros_cult_Staphylococcus aureus',
       'otros_cult_Streptococcus pneumoniae', 'otros_cult__Enterobacteria',
       'otros_cult__Fungi', 'otros_cult__Other bacteria', 'otros_cult__Virus',
       'bmr_etiologia', 'fenotipo_resistencia', 'Enterococcus',
       'Escherichia coli', 'Klebsiella pneumoniae', 'NEGATIVE',
       'Pseudomonas aeruginosa', 'Staphylococcus aureus',
       'Streptococcus pneumoniae', '_Enterobacteria', '_Fungi',
       '_Other bacteria', 'resultado_hemo'],
      dtype='object')


/tmp/ipykernel_1881424/1150911312.py:27: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  all_urg.rename(columns=lambda c: c.replace('otros_cult_', ''))


,Enterococcus,Escherichia coli,Klebsiella pneumoniae,Pseudomonas aeruginosa,Staphylococcus aureus,Streptococcus pneumoniae,_Enterobacteria,_Fungi,_Other bacteria,_Virus,bmr_etiologia,fecha_ingreso_urgencias,fenotipo_resistencia,person_id,resultado_hemo,NEGATIVE,all_cult_org
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,2021-04-22,"(0.0,)",1,"(NEGATIVE,)",1.0,NEGATIVE
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,2021-05-07,"(0.0,)",2,"(Enterococcus,)",1.0,Enterococcus
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,2021-06-04,"(0.0,)",3,"(NEGATIVE,)",1.0,NEGATIVE
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,2021-06-22,"(0.0,)",4,"(NEGATIVE,)",1.0,NEGATIVE
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,2021-03-24,"(0.0,)",5,"(NEGATIVE,)",1.0,NEGATIVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3908,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,2023-01-18,"(1.0,)",3909,"(Escherichia coli,)",0.0,Escherichia coli
3909,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,2023-04-23,"(1.0,)",3910,"(Escherichia coli,)",1.0,Escherichia coli
3910,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,2023-04-10,"(1.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0)",3911,"(Klebsiella pneumoniae,)",0.0,Klebsiella pneumoniae
3911,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0,2023-06-19,"(1.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0)",3912,"(Escherichia coli,)",1.0,Escherichia coli


### Build focus-of-infection code map

Translate numeric infection focus codes to human-readable labels for display and feature engineering.

In [44]:
print(tbl_codes2names[tbl_codes2names["variable"].str.contains("foco")][["value", "name"]])

    value                             name
181     1                  foco | pulmonar
182     2            foco | intraabdominal
183     3                    foco | biliar
184     4                  foco | urinario
185     5            foco | cardiovascular
186     6                      foco | piel
187     7  foco | sistema nervioso central
188     8            foco | catéter venoso
189     9  foco | vías altas respiratorias
190    10            foco | osteoarticular
191    11                   foco | genital
192    12               foco | desconocido


In [45]:
# Build a dict mapping numeric focus-of-infection codes → human-readable labels.
# The 'name' field in tbl_codes2names contains 'foco | label' — strip the prefix.
foco_map = (
    tbl_codes2names[tbl_codes2names["variable"].str.contains("foco")][["value", "name"]]
    .apply(lambda x: x.str.replace("foco | ", ""), axis=1)
)
foco_map = {int(x): v for x, v in foco_map.set_index("value")["name"].to_dict().items()}

### Export co-infection cases for clinician review (DISABLED)

This block would export patients with multiple organisms in their blood culture to Excel for manual resolution. It is disabled because `coinf_res_dict` already encodes those decisions.

In [46]:
# NOTE: DISABLED — the entire cell is a commented-out block.
# This code would export co-infection patients (those with >1 organism in resultado_hemo)
# to an Excel file for manual clinician review. It was used during the co-infection resolution
# QC phase and is no longer needed once coinf_res_dict (cell above) is finalised.
"""co_inf_patients = pd.merge(tbl_sepsis, hemo_urg_pivoted)[["person_id", "foco", "resultado_hemo"]]
co_inf_patients = co_inf_patients[co_inf_patients["resultado_hemo"].apply(lambda x: len(x) > 1)]
co_inf_patients["foco"] = co_inf_patients["foco"].map(foco_map)
co_inf_patients["resultado_hemo"] = co_inf_patients["resultado_hemo"].apply(lambda x: ", ".join(x)).str.replace(" (organismo)", "")
co_inf_patients.to_excel(os.path.join(save_location, "co_infection_patients.xlsx"), index=False)"""

'co_inf_patients = pd.merge(tbl_sepsis, hemo_urg_pivoted)[["person_id", "foco", "resultado_hemo"]]\nco_inf_patients = co_inf_patients[co_inf_patients["resultado_hemo"].apply(lambda x: len(x) > 1)]\nco_inf_patients["foco"] = co_inf_patients["foco"].map(foco_map)\nco_inf_patients["resultado_hemo"] = co_inf_patients["resultado_hemo"].apply(lambda x: ", ".join(x)).str.replace(" (organismo)", "")\nco_inf_patients.to_excel(os.path.join(save_location, "co_infection_patients.xlsx"), index=False)'

## Assemble the modelling dataset

Merge patient characteristics, comorbidities, risk factors, cultures, and outcomes into a single training-ready dataframe.

In [47]:
# Sequential left-joins assemble all processed tables around the patient master record.
# Using left joins ensures all patients are retained even if they have no record in a sub-table.
df_merged = df_pacientes.merge(tbl_comorbilidad, on=['person_id', 'fecha_ingreso_urgencias'], how='left')
df_merged = df_merged.merge(tbl_factores_riesgo_bmr, on=['person_id', 'fecha_ingreso_urgencias'], how='left')
df_merged = df_merged.merge(tbl_sepsis, on=['person_id', 'fecha_ingreso_urgencias'], how='left')
df_merged = df_merged.merge(tbl_signos, on=['person_id', 'fecha_ingreso_urgencias'], how='left')
df_merged = df_merged.merge(tbl_sintomas_complete, on=['person_id', 'fecha_ingreso_urgencias'], how='left')

# Track which columns existed before joining history tables (to selectively fill NaN below)
orig_cols = df_merged.columns.tolist()

# Infection history is patient-level (join on person_id only — no date needed)
df_merged = df_merged.merge(tbl_infecciones_complete, on=['person_id'], how='left')

# Colonisation history is also patient-level
df_merged = df_merged.merge(colo_prev_pivoted, on=['person_id'], how='left')

# Prior antibiotic use is joined on both patient and admission date
df_merged = df_merged.merge(antib_prev_to_merge, on=['person_id', 'fecha_ingreso_urgencias'], how='left')

# Fill missing values with 0 for all new binary/count columns (patients with no record = no exposure)
new_cols = [c for c in df_merged.columns if c not in orig_cols]
df_merged[new_cols] = df_merged[new_cols].fillna(0)

# Attach hemoculture outcome columns (resultado_hemo, BMR flag, resistance phenotype)
df_merged = pd.merge(
    df_merged,
    hemo_urg_pivoted[["person_id", "resultado_hemo", "bmr_etiologia", "fenotipo_resistencia"]],
    on=['person_id'], how='left'
)

# Attach combined culture dominant organism label
df_merged = pd.merge(df_merged, all_urg_comb[["person_id", "all_cult_org"]], on=['person_id'], how='left')

In [65]:
df_merged['resultado_hemo'].value_counts()

resultado_hemo
(NEGATIVE,)                    2072
(Escherichia coli,)             850
(Klebsiella pneumoniae,)        210
(_Other bacteria,)              198
(_Enterobacteria,)              176
(Staphylococcus aureus,)        171
(Streptococcus pneumoniae,)     106
(Pseudomonas aeruginosa,)        84
(Enterococcus,)                  39
(_Fungi,)                         7
Name: count, dtype: int64

### Aggregate per-organism event totals

For each organism group, sum all related binary/count columns across all sources into a single `_total` column.

In [66]:
# For each organism group, sum all related binary/count columns across infection and colonisation history.
# Creates e.g. 'Escherichia coli_total' = total events from all sources for that organism.
for org in set(organism_codes_map.values()):
    cols = [col for col in df_merged.columns if org in col]
    df_merged[f"{org}_total"] = df_merged[cols].sum(axis=1)

### Collapse cancer and hepatopathy dummies into aggregate flags

Replace multiple one-hot encoded cancer/hepatopathy columns with single binary presence flags.

In [67]:
def merge_columns(df_to_clean, column_list, new_col_name):
    """Sum a list of columns into a single aggregated column and drop the originals."""
    df_to_clean[new_col_name] = df_to_clean[column_list].sum(axis=1).astype(int)
    clean_df = df_to_clean.drop(columns=column_list)
    return clean_df

# Collect all one-hot encoded cancer and hepatopathy dummy columns created earlier
hepatic_cols = [c for c in df_merged.columns if "hepatopatia" in c]
tumor_cols = [c for c in df_merged.columns if "cancer" in c]

# Replace individual cancer/hepatopathy dummies with a single aggregate count column
for new_name, col_list in {"hepatopatias_totales": hepatic_cols, "canceres_totales": tumor_cols}.items():
    try:
        df_merged = merge_columns(df_merged, col_list, new_name)
    except Exception as e:
        print(f"Error merging columns {col_list} into {new_name}: {e}")

# Binary presence/absence flags derived from the aggregate counts
df_merged["canceres_si_no"] = np.where(df_merged["canceres_totales"] > 0, 1, 0)
df_merged["hepatopatias_si_no"] = np.where(df_merged["hepatopatias_totales"] > 0, 1, 0)

# Drop the raw aggregate count columns (binary flags are sufficient for modelling)
df_merged = df_merged.drop(columns=["canceres_totales", "hepatopatias_totales"])

# Replace 0.0 with NaN for days-since-last-antibiotic (0 is not a valid duration; means data missing)
df_merged["dias_ultimo_antib"] = df_merged["dias_ultimo_antib"].replace(0.0, np.nan)

### Explode co-infection tuples and map categorical fields

Expand the multi-organism `resultado_hemo` tuple into individual rows (one per organism) to enable per-organism label assignment.

In [68]:
df_merged['resultado_hemo'].unique()

array([('NEGATIVE',), ('Enterococcus',), ('_Other bacteria',),
       ('Pseudomonas aeruginosa',), ('_Enterobacteria',),
       ('Klebsiella pneumoniae',), ('Staphylococcus aureus',),
       ('Escherichia coli',), ('Streptococcus pneumoniae',), ('_Fungi',)],
      dtype=object)

In [69]:
# Explode 'resultado_hemo' so each organism in a co-infection tuple becomes its own row.
# This produces one row per (patient, organism) pair, enabling per-organism label assignment.
df_expanded = df_merged.explode("resultado_hemo").reset_index(drop=True)

# Map numeric focus codes to human-readable labels (e.g. 1 → 'urinario')
df_expanded["foco"] = df_expanded["foco"].map(foco_map)

# Convert BMR flag to descriptive string for clarity
df_expanded["bmr_etiologia"] = np.where(df_expanded["bmr_etiologia"] == 1.0, "BMR resistente", "NEGATIVE")

## Feature enrichment (synthetic)

In [70]:
# ── Synthetic features derived from existing columns ────────────────────────

# Total colonisation burden: sum of all colo_ binary columns
df_expanded["colonizacion_total_grouped"] = df_expanded.filter(like="colo_").sum(axis=1)

# Early recurrence flag: 1 if the last infection occurred within 30 days before admission
df_expanded["recurrencia_precoz"] = (df_expanded["tiempo_ultima"] < 30).astype(int)

# Infection density: infections per day of follow-up (adds 1 to avoid division by zero)
df_expanded["densidad_inf"] = df_expanded["num_inf_previas"] / (df_expanded["tiempo_ultima"] + 1)

# Composite high-dependency flag: patient lives in a care home OR is on permanent dialysis
df_expanded["residencia_dialisis"] = (
    (df_expanded["paciente_residencia"] == 1) |
    (df_expanded["hemodialisis_permanente"] == 1)
).astype(int)

# Device burden: total number of invasive medical devices present
dispositives = [
    "cateter_venoso", "sonda_urinaria",
    "sonda_nasogastrica", "derivacion_ventriculoper",
    "valvula_prot_cardiaca", "portador_otros_disposit"
]
df_expanded["carga_dispositivos"] = df_expanded[dispositives].sum(axis=1)

# Immunosuppression risk count: number of distinct immunocompromising conditions present
inmuno = [
    "inmunosupresion", "sida", "linfoma",
    "leucemia", "neoplasia"
]
df_expanded["total_inmunoriesgo_cat"] = (df_expanded[inmuno].gt(0).sum(axis=1)).astype(int)

# NOTE: DISABLED — raw (un-capped) immunosuppression sum; replaced by the categorical version above
#df_expanded["total_inmunoriesgo_raw"] = df_expanded[inmuno].sum(axis=1)

organs = ["erc", "hepatopatias_si_no", "insuficiencia_cardiaca"]

# NOTE: DISABLED — organ failure count features; excluded from current experiments
#df_expanded["riesgo_organos_cat"] = (df_expanded[organs].gt(0).sum(axis=1)).astype(int)
#df_expanded["riesgo_organos_raw"] = df_expanded[organs].sum(axis=1)

# NOTE: DISABLED — urinary and respiratory syndrome composite scores.
# Kept as reference; may be re-enabled if foco-specific models are developed.
"""
df_expanded["total_urinario"] = (
    df_expanded[
        ["sintoma_disuria",
         "sintoma_síndrome de disuria - polaquiuria",
         "sonda_urinaria"]
    ]
    .fillna(0)
    .sum(axis=1)
)

df_expanded["total_respiratorio"] = (
    df_expanded[
        ["sintoma_tos",
         "sintoma_dificultad para respirar",
         "hipoxemia"]
    ]
    .fillna(0)
    .sum(axis=1)
)
"""

'\ndf_expanded["total_urinario"] = (\n    df_expanded[\n        ["sintoma_disuria",\n         "sintoma_síndrome de disuria - polaquiuria",\n         "sonda_urinaria"]\n    ]\n    .fillna(0)\n    .sum(axis=1)\n)\n\ndf_expanded["total_respiratorio"] = (\n    df_expanded[\n        ["sintoma_tos",\n         "sintoma_dificultad para respirar",\n         "hipoxemia"]\n    ]\n    .fillna(0)\n    .sum(axis=1)\n)\n'

### Inspect colonisation feature columns

Exploratory cell to list all `colo_` prefixed columns in df_expanded.

In [71]:
# Exploratory: list all colonisation feature columns present in df_expanded
[x for x in df_expanded.columns if "colo" in x]

['colo_Escherichia coli_binary',
 'colo_Pseudomonas aeruginosa_binary',
 'colo_Staphylococcus aureus_binary',
 'colo_Klebsiella pneumoniae_binary',
 'colo__Other bacteria_binary',
 'colo__Enterobacteria_binary',
 'colo__Virus_binary',
 'colo_Enterococcus_binary',
 'colo__Fungi_binary',
 'colo_Streptococcus pneumoniae_binary',
 'colonizacion_total_grouped']

### Check organ failure indicator distribution

Exploratory cell to verify value counts for the three organ-failure columns.

In [72]:
# Exploratory: check value distribution of organ-failure indicator columns
df_expanded[organs].value_counts()

erc  hepatopatias_si_no  insuficiencia_cardiaca
0    0                   0                         2502
2    0                   0                          511
0    0                   1                          415
2    0                   1                          213
0    1                   0                          188
2    1                   0                           37
0    1                   1                           28
2    1                   1                           19
Name: count, dtype: int64

### Group bacterias

In [73]:
# Map specific organism names to broader Gram-stain based groups for the grouped target variable.
# Organisms not listed here keep their original name (via .fillna in the apply below).
micro_map = {
    "Escherichia coli": "Bacilo gram-",       # Gram-negative rod (Enterobacteriaceae)
    "Klebsiella pneumoniae": "Bacilo gram-",  # Gram-negative rod (Enterobacteriaceae)
    "_Enterobacteria": "Bacilo gram-",         # Other Gram-negative rods
    "Pseudomonas aeruginosa": "Bacilo gram-",  # Gram-negative rod (non-fermenter)
    "Staphylococcus aureus": "Coco gram+",    # Gram-positive coccus
    "Streptococcus pneumoniae": "Coco gram+", # Gram-positive coccus
    # Organisms not listed (e.g. Enterococcus, _Fungi, NEGATIVE) remain unchanged
}

# Apply the mapping; unmapped values fall back to the original resultado_hemo label
df_expanded["resultado_hemo_grouped"] = df_expanded["resultado_hemo"].map(micro_map).fillna(df_expanded["resultado_hemo"])

### Create grouped infection and colonisation columns for the Gram-stain target

Aggregate species-level previous infection and colonisation binary columns into Gram-stain group totals.

In [74]:
# Create aggregate previous-infection and colonisation columns for the grouped bacterial target.
# These sum up all species-level binary flags that belong to the same Gram-stain group.
df_expanded["Inf_Bacilo gram-"] = 0
df_expanded["Inf_Coco gram+"] = 0
df_expanded["Colo_Bacilo gram-"] = 0
df_expanded["Colo_Coco gram+"] = 0

# Mapping: new column prefix → source column prefix
# 'Inf_' reads from infection binary columns (no prefix in column name)
# 'Colo_' reads from colonisation binary columns (prefixed with 'Colo_')
new_colmap = {
    "Inf_": "",
    "Colo_": "colo_"
}
for species, group in micro_map.items():
    for newcolpref, datapref in new_colmap.items():
        newcol = newcolpref + group
        col_name = datapref + species + "_binary"
        if col_name in df_expanded.columns:
            df_expanded[newcol] += df_expanded[col_name]   # accumulate across all species in the group

print(df_expanded)

      person_id fecha_ingreso_urgencias fecha_nacimiento  edad  sexo  \
0             1              2021-04-22       1939-07-23    81     0   
1             2              2021-05-07       1939-07-23    81     0   
2             3              2021-06-04       1939-07-23    81     0   
3             4              2021-06-22       1939-07-23    81     0   
4             5              2021-03-24       1944-08-20    76     1   
...         ...                     ...              ...   ...   ...   
3908       3909              2023-01-18       1979-09-02    43     1   
3909       3910              2023-04-23       1974-05-25    48     1   
3910       3911              2023-04-10       1944-02-06    79     0   
3911       3912              2023-06-19       1967-05-03    56     0   
3912       3913              2023-06-26       1989-03-26    34     1   

      codigo_postal  mujer_gestante  mayor_65  paciente_residencia  \
0              8036               0         1                    

# Create BMR + phenotype prediction dataframe

### Check resist phenotype distribution

In [80]:
# Build a dict mapping numeric resistance phenotype codes → human-readable labels
phenomap = {
    float(x["value"]): x["name"]
    for x in tbl_codes2names[tbl_codes2names["variable"] == "fenotipo_resistencia"][["value", "name"]]
    .to_dict(orient="records")
}

# Exploratory: check the distribution of resistance phenotype tuples in the full dataset
df_expanded["fenotipo_resistencia"].apply(
    lambda x: tuple(phenomap.get(n, "NEGATIVE").split(" resistente a ")[-1] for n in x)
).value_counts()

fenotipo_resistencia
(NEGATIVE,)                                                                                                                                                                                3095
(amoxicilina/clavulánico,)                                                                                                                                                                  138
(ampicilina o penicilina,)                                                                                                                                                                   95
(ciprofloxacino,)                                                                                                                                                                            64
(NEGATIVE, NEGATIVE)                                                                                                                                                                         51
                   

### Build df_cefalosporinas — cephalosporin resistance dataset

Decode resistance phenotype codes to drug names as a first step toward the cephalosporin-specific prediction target.

In [98]:
# Create a deep copy of df_expanded dedicated to cephalosporin resistance prediction
df_cefalosporinas = copy.deepcopy(df_expanded)

# Decode numeric phenotype codes → drug names using phenomap; keep only the drug part after ' resistente a '
df_cefalosporinas["fenotipo_resistencia"] = df_cefalosporinas["fenotipo_resistencia"].apply(
    lambda x: tuple(phenomap.get(n, "NEGATIVE").split(" resistente a ")[-1] for n in x)
)
# Second pass handles any residual 'X resistente a Y' strings left after first decode
df_cefalosporinas["fenotipo_resistencia"] = df_cefalosporinas["fenotipo_resistencia"].apply(
    lambda x: tuple(z.split(" resistente a ")[-1] if not pd.isna(z) else z for z in x)
)
df_cefalosporinas["fenotipo_resistencia"].value_counts()

fenotipo_resistencia
(NEGATIVE,)                                                                                                                                                                                3095
(amoxicilina/clavulánico,)                                                                                                                                                                  138
(ampicilina o penicilina,)                                                                                                                                                                   95
(ciprofloxacino,)                                                                                                                                                                            64
(NEGATIVE, NEGATIVE)                                                                                                                                                                         51
                   

### Map resistance phenotypes to pharmacological families

Translate decoded drug names to their pharmacological class using `antibdict` and `antimicrobial_groups`.

In [99]:
# Translation dict: Spanish/abbreviated drug name in data → standardised English name
antibdict = {
    "amoxicilina/clavulánico":     "amoxicillin and beta-lactamase inhibitor",
    "ciprofloxacino":               "ciprofloxacin",
    "ceftrixona o cefotaxima":      "ceftriaxone",
    "ceftazidima":                  "ceftazidime",
    "piperacilina/tazobactam":      "piperacillin and beta-lactamase inhibitor",
    "cefepima":                     "cefepime",
    "ampicilina o penicilina":      "ampicillin",
    "meticilina":                   "cloxacillin",
    "meropenem":                    "meropenem",
    "ceftazidima/avibactam":        "ceftazidime and beta-lactamase inhibitor",
    "ceftolozano/tazobactam":       "ceftolozane and beta-lactamase inhibitor",
    "vancomicina":                  "vancomycin"
}
# NOTE: DISABLED — diagnostic line to verify all antibdict values resolve to a known drug family
#[antimicrobial_groups.get(x, x) for x in antibdict.values()]

# Build final mapping: drug name found in fenotipo_resistencia → pharmacological family
fenotipo_groups = {x: antimicrobial_groups.get(v, v) for x, v in antibdict.items()}

# Replace each drug name in the phenotype tuple with its pharmacological family
df_cefalosporinas["fenotipo_resistencia"] = df_cefalosporinas["fenotipo_resistencia"].apply(
    lambda x: tuple(fenotipo_groups.get(z, "NEGATIVE") for z in x)
)

In [100]:
fenotipo_groups

{'amoxicilina/clavulánico': 'Penicilinas',
 'ciprofloxacino': 'Quinolonas',
 'ceftrixona o cefotaxima': 'Cefalosporinas 3 gen',
 'ceftazidima': 'Cefalosporinas 3 gen',
 'piperacilina/tazobactam': 'Penicilinas',
 'cefepima': 'Cefalosporinas 4 gen',
 'ampicilina o penicilina': 'Penicilinas',
 'meticilina': 'Penicilinas',
 'meropenem': 'Carbapenemas',
 'ceftazidima/avibactam': 'Cefalosporinas 3 gen',
 'ceftolozano/tazobactam': 'Cefalosporinas 3 gen',
 'vancomicina': 'Glicopéptidos'}

### Derive cephalosporin resistance label

Assign each row a three-class label: resistant to 3rd/4th gen cephalosporins, resistant to other drugs, or negative.

In [101]:
# Derive the cephalosporin resistance label:
#   RESIST_CEFALOSPORINAS_3a_4a → any 3rd/4th gen cephalosporin resistance present and not NEGATIVE
#   OTHER                       → resistant to something else (but not 3rd/4th gen cephalosporins)
#   NEGATIVE                    → all phenotypes are NEGATIVE (susceptible / not tested)
df_cefalosporinas["resistente_cefalosporina"] = df_cefalosporinas["fenotipo_resistencia"].apply(
    lambda x: "RESIST_CEFALOSPORINAS_3a_4a"
    if (any("Cefalosporina" in z for z in x) and not x[0] == "NEGATIVE")
    else ("OTHER" if not all(z == "NEGATIVE" for z in x) else "NEGATIVE")
)
# NOTE: DISABLED — second formulation of the same logic; kept for reference
#df_cefalosporinas["resistente_cefalosporina"] =

In [102]:
df_cefalosporinas["resistente_cefalosporina"].unique()

array(['NEGATIVE', 'OTHER', 'RESIST_CEFALOSPORINAS_3a_4a'], dtype=object)

In [103]:
df_cefalosporinas.shape

(3913, 188)

### Save cephalosporin-specific dataset

Write the df_cefalosporinas dataset to disk for downstream cephalosporin resistance modelling.

In [104]:
# Assign uniform sample weight (1) — no class-weighting applied for this dataset export
df_cefalosporinas["sample_weight"] = 1

# Persist the cephalosporin-focused dataset to disk for downstream modelling
# df_cefalosporinas.to_csv("/home/pmata/mepram_data/df_cefalosporinas_bmr.csv")

### NOTE: Potentially broken mapping step (comment cell if fenotipo_resistencia is a tuple)

Applies `fenotipo_groups` as a scalar map to the phenotype column. Behaviour depends on whether the column holds strings or tuples at this point.

In [105]:
# NOTE: This cell applies fenotipo_groups (drug → family map) directly to the whole column using .map(),
# which only works if fenotipo_resistencia contains scalar strings (not tuples).
# This is inconsistent with previous cells that store tuples — this line may raise errors or produce NaN
# if the column still contains tuples at this point. Kept as-is for reproducibility.
# df_cefalosporinas["fenotipo_resistencia"] = df_cefalosporinas["fenotipo_resistencia"].map(fenotipo_groups).fillna("NEGATIVE")

In [106]:
df_cefalosporinas["fenotipo_resistencia"].value_counts()

fenotipo_resistencia
(NEGATIVE,)                                                                                                  3095
(Penicilinas,)                                                                                                271
(Penicilinas, Penicilinas)                                                                                     71
(Quinolonas,)                                                                                                  64
(NEGATIVE, NEGATIVE)                                                                                           51
                                                                                                             ... 
(Penicilinas, Penicilinas, Quinolonas, Quinolonas)                                                              1
(NEGATIVE, NEGATIVE, Penicilinas, Penicilinas)                                                                  1
(Cefalosporinas 3 gen, Cefalosporinas 4 gen, Cefalosporinas 3 gen, 

## Build df_resist — per-phenotype exploded dataset for BMR modelling

Explode the resistance phenotype column so each row represents one patient–phenotype pair. Assign inverse-frequency sample weights.

In [ ]:
# Create a per-resistance-phenotype exploded dataset for the BMR/resistance multi-class task.
# Each row in df_resist represents one (patient, phenotype) pair.
df_resist = df_expanded.explode("fenotipo_resistencia").reset_index(drop=True) # names to integeres again

# Compute inverse-frequency sample weights so patients with many phenotype rows don't dominate training
person_freq = df_resist["person_id"].value_counts()
df_resist["sample_weight"] = df_resist["person_id"].map(1 / person_freq)

In [109]:
df_resist.shape

(5044, 188)

### Decode resistance phenotype codes

Map numeric codes to drug names, then strip the 'resistente a' prefix to get clean drug labels.

In [112]:
# Decode numeric phenotype codes to human-readable drug names using phenomap
df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].map(phenomap)

# Keep only the drug name (the part after ' resistente a ') for cleaner labels
df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].apply(
    lambda x: x.split(" resistente a ")[-1] if not pd.isna(x) else x
)

### Re-define drug translation dicts (standalone execution support)

Duplicate of cell 74 — ensures this cell can run even if the kernel was restarted.

In [113]:
# Re-define the same translation dicts here (duplicate of cell 74) to ensure this cell can run
# independently even if the kernel was restarted or cell 74 was skipped.
antibdict = {
    "amoxicilina/clavulánico":     "amoxicillin and beta-lactamase inhibitor",
    "ciprofloxacino":               "ciprofloxacin",
    "ceftrixona o cefotaxima":      "ceftriaxone",
    "ceftazidima":                  "ceftazidime",
    "piperacilina/tazobactam":      "piperacillin and beta-lactamase inhibitor",
    "cefepima":                     "cefepime",
    "ampicilina o penicilina":      "ampicillin",
    "meticilina":                   "cloxacillin",
    "meropenem":                    "meropenem",
    "ceftazidima/avibactam":        "ceftazidime and beta-lactamase inhibitor",
    "ceftolozano/tazobactam":       "ceftolozane and beta-lactamase inhibitor",
    "vancomicina":                  "vancomycin"
}
# NOTE: DISABLED — diagnostic check to verify English names resolve to known families
#[antimicrobial_groups.get(x, x) for x in antibdict.values()]

fenotipo_groups = {x: antimicrobial_groups.get(v, v) for x, v in antibdict.items()}

# Map decoded drug names → pharmacological families; unmapped values → NEGATIVE
df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].map(fenotipo_groups).fillna("NEGATIVE")
# NOTE: DISABLED — alternative without fillna; would leave unmapped values as NaN
#df_resist["fenotipo_resistencia"] = df_resist["fenotipo_resistencia"].map(fenotipo_groups)

In [114]:
df_resist["fenotipo_resistencia"].value_counts()

fenotipo_resistencia
NEGATIVE                3275
Penicilinas              855
Cefalosporinas 3 gen     411
Quinolonas               319
Cefalosporinas 4 gen     171
Carbapenemas              11
Glicopéptidos              2
Name: count, dtype: int64

### Identify high-missingness columns

Find all columns with more than 5.2% missing values and show how the BMR class balance changes after excluding them.

In [116]:
# Identify columns with more than 5.2% missing values (chosen threshold balances data completeness
# against dropping too many features). These columns would be excluded before model training.
dropped_for_na = []
for col in df_resist:
    na_per = df_resist[col].isna().mean()
    if na_per > 0.052:
        dropped_for_na.append(col)

# Compare BMR class distribution before and after dropping high-NA rows
print(df_resist["bmr_etiologia"].value_counts())
print(df_resist.drop(columns=dropped_for_na).dropna()["bmr_etiologia"].value_counts())
print(f"Dropped columns due to high NA: {dropped_for_na}")

bmr_etiologia
NEGATIVE          3215
BMR resistente    1829
Name: count, dtype: int64
bmr_etiologia
NEGATIVE          2792
BMR resistente    1625
Name: count, dtype: int64
Dropped columns due to high NA: ['causa_inmunosupresion', 'situacion_funcional_basal', 'cirugia_previa_con_implant', 'sofa', 'snc_glasgow', 'bilirrubina', 'proteina_c_reactiva', 'proteina_c_reactiva_recoded', 'frec_respiratoria', 'frec_respiratoria_recoded', 'dias_ultimo_antib']


### Derive cephalosporin resistance label and save df_resist

Add the binary cephalosporin resistance column and persist the resistance-focused dataset to disk.

In [117]:
# NOTE: DISABLED — would set uniform sample weight (no per-patient weighting)
#df_resist["sample_weight"] = 1

# Derive cephalosporin resistance binary label for the exploded resistance dataset
df_resist["resistente_cefalosporina"] = np.where(
    df_resist["fenotipo_resistencia"].str.contains("Cefalosporinas"),
    "RESIST_CEFALOSPORINAS_3a_4a",
    "NEGATIVE"
)

# Save the resistance-focused dataset to disk
# df_resist.to_csv(os.path.join(save_location, "df_resist_bmr_grouped.csv"), index=False)

### Exploratory: infection focus × organism cross-tabulation

Compute and display percentage breakdown of causative organism by infection focus, restricted to the main clinical categories.

In [118]:
# Exploratory: cross-tabulation of infection focus (foco) vs. bacteraemia organism (resultado_hemo).
# Excludes uncommon organisms and uncommon/atypical infection foci to focus on clinically relevant cases.
testdf = df_expanded[~df_expanded["resultado_hemo"].isin(["_Fungi", "_Other bacteria", "Enterococcus"])]
testdf = testdf[~testdf["foco"].isin([
    "piel", "catéter venoso", "vías altas respiratorias",
    "cardiovascular", "osteoarticular", "sistema nervioso central", "genital"
])]

# Raw counts: rows = infection focus, columns = causative organism
freq_foco_resultado = (
    testdf.groupby(["foco", "resultado_hemo"], observed=True)
    .size()
    .unstack(fill_value=0)
)

# Normalise by column (per organism) to show what % of each organism's cases come from each focus
freq_foco_resultado_pct = freq_foco_resultado.div(freq_foco_resultado.sum(axis=0), axis=1) * 100
freq_foco_resultado_pct.applymap(lambda v: f"{v:.1f}%")

/tmp/ipykernel_1881424/1401425270.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  freq_foco_resultado_pct.applymap(lambda v: f"{v:.1f}%")


resultado_hemo,Escherichia coli,Klebsiella pneumoniae,NEGATIVE,Pseudomonas aeruginosa,Staphylococcus aureus,Streptococcus pneumoniae,_Enterobacteria
foco,,,,,,,
biliar,12.7%,12.6%,2.3%,4.4%,1.1%,0.0%,7.8%
desconocido,7.1%,7.1%,13.1%,26.5%,33.7%,5.2%,12.7%
intraabdominal,12.8%,13.6%,10.8%,8.8%,5.4%,1.0%,18.1%
pulmonar,6.4%,6.6%,47.1%,26.5%,31.5%,91.8%,9.6%
urinario,60.9%,60.1%,26.6%,33.8%,28.3%,2.1%,51.8%


### Create "infected_yes_no" as a binary head for fenotype prediction

In [ ]:
# Binary infection label: 'POSITIVE' for any organism detected, 'NEGATIVE' for blood-culture-negative cases
df_expanded["infected_yes_no"] = np.where(df_expanded["resultado_hemo"] == "NEGATIVE", "NEGATIVE", "POSITIVE")

### Multi-hot encode fenotipo_resistencia for multi-label prediction

In [120]:
# Decode numeric phenotype codes and keep only the drug name (strip 'X resistente a ' prefix).
# Codes not found in phenomap are silently dropped (if n not in phenomap → excluded from tuple).
df_expanded["fenotipo_resistencia"] = df_expanded["fenotipo_resistencia"].apply(
    lambda x: tuple(phenomap[n].split(" resistente a ")[-1] for n in x if n in phenomap)
)

### Group by pharmacological family

In [126]:
# Map each decoded drug name to its pharmacological family.
# Drug names not present in fenotipo_groups are silently dropped.
df_expanded["fenotipo_resistencia"] = df_expanded["fenotipo_resistencia"].apply(
    lambda x: tuple(fenotipo_groups[n] for n in x if n in fenotipo_groups)
)

### Persist intermediate datasets

Write exploded datasets to disk so later steps can reload them without recomputing heavy joins.

In [127]:
# NOTE: DISABLED — these two lines were used to inspect unique focus values and their distribution.
# Left as comments for quick re-activation during debugging.
#df_expanded["foco"].unique()
#df_expanded["foco"]
#print(df_expanded["foco"].map(foco_map).value_counts())

# Distribution of causative organisms in the full exploded dataset
print(df_expanded["resultado_hemo"].value_counts())

# Shape with and without missing focus values
print("no NA:", df_expanded[~df_expanded["foco"].isna()].shape)
print("normal shape:", df_expanded.shape)

# Distribution after excluding rare/atypical infection foci
print("excluding minor focos: ",
    df_expanded[
        ~df_expanded["foco"].map(foco_map).isin([
            "catéter venoso", "vías altas respiratorias", "cardiovascular",
            "osteoarticular", "sistema nervioso central", "genital"
        ])
    ]["resultado_hemo"].value_counts()
)

resultado_hemo
NEGATIVE                    2072
Escherichia coli             850
Klebsiella pneumoniae        210
_Other bacteria              198
_Enterobacteria              176
Staphylococcus aureus        171
Streptococcus pneumoniae     106
Pseudomonas aeruginosa        84
Enterococcus                  39
_Fungi                         7
Name: count, dtype: int64
no NA: (3913, 187)
normal shape: (3913, 187)
excluding minor focos:  resultado_hemo
NEGATIVE                    2072
Escherichia coli             850
Klebsiella pneumoniae        210
_Other bacteria              198
_Enterobacteria              176
Staphylococcus aureus        171
Streptococcus pneumoniae     106
Pseudomonas aeruginosa        84
Enterococcus                  39
_Fungi                         7
Name: count, dtype: int64


### INCOMPLETE: grouped organism binary column

This cell was left unfinished — the assignment `df_expanded["prev_org_grouped_binary"] =` has no right-hand side.

In [ ]:
# Exploratory / INCOMPLETE — this cell has a syntax error (missing right-hand side of assignment)
# and was never finalised. The intent was to create a binary column indicating whether a patient
# had any previous infection with an organism in the grouped micro_map categories.
print([x for x in micro_map])
print([x for x in df_expanded.columns])
prev_org_bincols = [x for x in df_expanded.columns if any(col+"_binary")]
df_expanded["prev_org_grouped_binary"] =   # NOTE: incomplete — right-hand side was never defined

# Save the finished table

In [ ]:
# Deduplicate resistance phenotype lists (a patient may have the same phenotype recorded twice)
df_expanded["fenotipo_resistencia"] = df_expanded["fenotipo_resistencia"].apply(dedupe_labels)

# Binary cephalosporin resistance flag (simple version: any Cefalosporinas entry → resistant)
df_expanded["resistente_cefalosporina"] = df_expanded["fenotipo_resistencia"].apply(
    lambda x: "RESIST_CEFALOSPORINAS_3a_4a" if any("Cefalosporinas" in cat for cat in x) else "NEGATIVE"
)

# Multi-class cephalosporin resistance (also captures OTHER resistance patterns)
df_expanded["resistente_cefalosporina_multi"] = df_expanded["fenotipo_resistencia"].apply(
    lambda x: "RESIST_CEFALOSPORINAS_3a_4a"
    if (any("Cefalosporina" in z for z in x) and not x[0] == "NEGATIVE")
    else ("OTHER" if not all(z == "NEGATIVE" for z in x) else "NEGATIVE")
)

In [132]:
print(df_expanded["fenotipo_resistencia"].value_counts())
print(df_expanded["resistente_cefalosporina"].value_counts())
print(df_expanded["resistente_cefalosporina_multi"].value_counts())

fenotipo_resistencia
[]                                                                                     3152
[Penicilinas]                                                                           377
[Cefalosporinas 3 gen, Cefalosporinas 4 gen, Penicilinas, Quinolonas]                    88
[Penicilinas, Quinolonas]                                                                75
[Quinolonas]                                                                             67
[Cefalosporinas 3 gen, Cefalosporinas 4 gen, Quinolonas]                                 36
[Cefalosporinas 3 gen, Penicilinas, Quinolonas]                                          25
[Cefalosporinas 3 gen, Penicilinas]                                                      21
[Cefalosporinas 3 gen, Cefalosporinas 4 gen, Penicilinas]                                20
[Cefalosporinas 3 gen, Cefalosporinas 4 gen]                                             12
[Cefalosporinas 3 gen, Quinolonas]                         

### Inspect resistance phenotype distribution

Exploratory: show value counts of deduplicated resistance phenotype tuples.

In [133]:
# Exploratory: check the distribution of resistance phenotype combinations in the final dataset
df_expanded["fenotipo_resistencia"].value_counts()

fenotipo_resistencia
[]                                                                                     3152
[Penicilinas]                                                                           377
[Cefalosporinas 3 gen, Cefalosporinas 4 gen, Penicilinas, Quinolonas]                    88
[Penicilinas, Quinolonas]                                                                75
[Quinolonas]                                                                             67
[Cefalosporinas 3 gen, Cefalosporinas 4 gen, Quinolonas]                                 36
[Cefalosporinas 3 gen, Penicilinas, Quinolonas]                                          25
[Cefalosporinas 3 gen, Penicilinas]                                                      21
[Cefalosporinas 3 gen, Cefalosporinas 4 gen, Penicilinas]                                20
[Cefalosporinas 3 gen, Cefalosporinas 4 gen]                                             12
[Cefalosporinas 3 gen, Quinolonas]                         

### Inspect key outcome columns

Exploratory: display the four main outcome columns side-by-side for a final quality check.

In [134]:
# Exploratory: inspect the four key outcome columns side-by-side
df_expanded[["resultado_hemo", "resultado_hemo_grouped", "resistente_cefalosporina", "fenotipo_resistencia"]]

,resultado_hemo,resultado_hemo_grouped,resistente_cefalosporina,fenotipo_resistencia
0,NEGATIVE,NEGATIVE,NEGATIVE,[]
1,Enterococcus,Enterococcus,NEGATIVE,[]
2,NEGATIVE,NEGATIVE,NEGATIVE,[]
3,NEGATIVE,NEGATIVE,NEGATIVE,[]
4,NEGATIVE,NEGATIVE,NEGATIVE,[]
...,...,...,...,...
3908,Escherichia coli,Bacilo gram-,NEGATIVE,[Penicilinas]
3909,Escherichia coli,Bacilo gram-,NEGATIVE,[Penicilinas]
3910,Klebsiella pneumoniae,Bacilo gram-,RESIST_CEFALOSPORINAS_3a_4a,"[Cefalosporinas 3 gen, Cefalosporinas 4 gen, P..."
3911,Escherichia coli,Bacilo gram-,RESIST_CEFALOSPORINAS_3a_4a,"[Cefalosporinas 3 gen, Cefalosporinas 4 gen, P..."


### Save final modelling dataset

Write the complete preprocessed dataset to the standard output path.

In [ ]:
# Assign uniform sample weight of 1 (no per-patient weighting in the final output)
df_expanded["sample_weight"] = 1

# Save the final modelling-ready dataset to disk
# df_expanded.to_csv(os.path.join(save_location, "df_merged_full_multilabel_grouped.csv"), index=False)

# Process bacthecom data

In [ ]:
import msoffcrypto
import io
import pandas as pd

# Password to decrypt the password-protected BACTHECOM Excel file.
# Replace 'placeholder' with the actual password before running.
password = "placeholder"

# Open and decrypt the file using the msoffcrypto library
with open("/home/pmata/mepram_data/bacthecom_hc_urgencias_v2_completo.xlsx", "rb") as file:
    office_file = msoffcrypto.OfficeFile(file)
    office_file.load_key(password=password)
    decrypted = io.BytesIO()        # in-memory buffer to hold decrypted bytes
    office_file.decrypt(decrypted)

bacthecom_df = pd.read_excel(decrypted)

# Identify columns shared between BACTHECOM and the main resistance dataset (for alignment/validation)
shared_cols = [col for col in bacthecom_df.columns if col in df_resist.columns]
bact_df_shared = bacthecom_df.copy()[shared_cols]

### Load and filter BACTHECOM Excel file

Load the external BACTHECOM (Spanish bacteraemia cohort) Excel and retain only the columns shared with the main resistance dataset.

In [ ]:
# Exploratory: check the organism distribution in df_expanded after excluding rare categories
# (Enterococcus, fungi, and other rare bacteria are filtered out to focus on the main pathogens)
df_expanded[~df_expanded["resultado_hemo"].isin([
    "Enterococcus",
    "_Fungi",
    "_Other bacteria",
])]["resultado_hemo"].value_counts()

# Optional: Data analysis

### Plot results from hierarchical_model_train_rfecv.py execution in hpc

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import f1_score
# binary_optuna_stacking_cef_normal_20260127114340
results_root = Path(
    "/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/binary_optuna_cef_new_20260202172017/"
)

subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None


for subset_dir in subset_dirs:
    # --- detectar modo por nombre de archivo ---
    binary_cm = subset_dir / "binary_confusion_matrix.csv"
    multiclass_cm = subset_dir / "multiclass_confusion_matrix.csv"

    if binary_cm.exists():
        mode = "binary"
        cm_path = binary_cm
    if multiclass_cm.exists():
        mode = "multiclass"
        cm_path = multiclass_cm
    if binary_cm.exists() or multiclass_cm.exists():
        if binary_cm.exists() and multiclass_cm.exists():
            mode= "hierarchical"
            print("THIS IS A HIERARCHICAL TEST")
    else:
        print(f"No confusion matrix found in {subset_dir}")
        continue

    predictions_dirs = [f for f in subset_dir.iterdir() if "predictions.csv" in f.name]
    if len(predictions_dirs) == 1:
        predictions_df = pd.read_csv(predictions_dirs[0])
    elif len(predictions_dirs) > 1:
        raise ValueError("MULTIPLE PREDICTIONS CSV FOUND: ", str(predictions_dirs))
    else:
        predictions_df = None
        print("NO PREDICTION DIR FOUND")

    if predictions_df is not None:
        coldict = {
            "binary": {"true": "true_binary_label", "pred": "binary_pred"},
            "hierarchical": {"true": "true_fenotipo", "pred": "hierarchical_pred"}
        }
        if mode == "hierarchical":
            predictions_df = predictions_df[(predictions_df["true_fenotipo"] != "NEGATIVE") & (predictions_df["hierarchical_pred"] != "NEGATIVE")]
            print(predictions_df)
            f1_score_m = f1_score(
                predictions_df[coldict[mode]["true"]],
                predictions_df[coldict[mode]["pred"]],
                average="micro",
                sample_weight=predictions_df.get("sample_weight", 1),
            )
            print("f1_score: ", f1_score_m)

    df = load_confusion_df(cm_path)
    if df is None:
        print(f"Empty confusion matrix in {subset_dir}")
        continue

    # limpieza de etiquetas (solo relevante para multiclase)
    if mode == "multiclass":
        df = df.rename(
            columns=lambda x: x.split("resistente a ")[-1],
            index=lambda x: x.split("resistente a ")[-1],
        )

    # --- cargar summary ---
    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    summary["f1_score_micro"] = f1_score_m
    if mode == "binary":
        suffix = (
            f"\nBinary macro ROC-AUC: {summary.get('binary_roc_auc'):.3f}"
            if summary.get("binary_roc_auc") is not None
            else ""
        )
        cmap = "Blues"
        title = f"{subset_dir.name} – Binary classification{suffix}"

    else:
        suffix = (
            f"\nMulticlass macro ROC-AUC: {summary.get('multiclass_macro_auc'):.3f}"
            if summary.get("multiclass_macro_auc") is not None
            else ""
        )
        cmap = "Greens"
        title = f"{subset_dir.name} – Multiclass classification{suffix}"

    # --- plot ---
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(df, annot=True, fmt=".2f", cmap=cmap, ax=ax)

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")

    plt.tight_layout()
    plt.show()

    break  # quita esto si quieres todos los subsets
else:
    print("subset_dirs is empty")


In [ ]:
aucs, f1s = [], []
for root_dir in Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/").iterdir():

    if not "binary_optuna_cefalosporinas" in root_dir.name and not "other" in root_dir.name:
        continue
    subset_dirs = sorted(
        d for d in root_dir.iterdir()
        if d.is_dir() and d.name.startswith("rfecv_")
    )
    
    def load_confusion_df(path: Path) -> pd.DataFrame | None:
        return pd.read_csv(path, index_col=0) if path.exists() else None


    for subset_dir in subset_dirs:
        # --- detectar modo por nombre de archivo ---
        binary_cm = subset_dir / "binary_confusion_matrix.csv"
        multiclass_cm = subset_dir / "multiclass_confusion_matrix.csv"

        if binary_cm.exists():
            mode = "binary"
            cm_path = binary_cm
        elif multiclass_cm.exists():
            mode = "multiclass"
            cm_path = multiclass_cm
        else:
            print(f"No confusion matrix found in {subset_dir}")
            continue

        df = load_confusion_df(cm_path)
        if df is None:
            print(f"Empty confusion matrix in {subset_dir}")
            continue

        # limpieza de etiquetas (solo relevante para multiclase)
        if mode == "multiclass":
            df = df.rename(
                columns=lambda x: x.split("resistente a ")[-1],
                index=lambda x: x.split("resistente a ")[-1],
            )

        # --- cargar summary ---
        summary_path = subset_dir / "summary.json"
        summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    print(root_dir.name)
    print(summary.get("binary_model"), summary.get("binary_roc_auc"))


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Point this to the run output root (e.g., outputs/binary_optuna_YYYYMMDDHHMMSS)
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/binary_optuna_cefalosporinas_dropother_20251217120107/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_bin")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

print(subset_dirs)
for subset_dir in subset_dirs:
    preds_path = subset_dir / "binary_predictions.csv"
    if not preds_path.exists():
        print(f"Missing predictions: {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    classes = sorted(set(preds["true_binary_label"]) | set(preds["binary_pred"]))

    binary_df = load_confusion_df(subset_dir / "binary_confusion_matrix.csv")
    if binary_df is None:
        y_true = preds["true_binary_label"]
        y_pred = preds["binary_pred"]
        cm = confusion_matrix(y_true, y_pred, labels=classes)
        binary_df = pd.DataFrame(
            cm,
            index=[f"true_{lbl}" for lbl in classes],
            columns=[f"pred_{lbl}" for lbl in classes],
        )
    for col in binary_df:
        binary_df[col] = binary_df[col].astype(int)

    # Load summary for metrics
    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    auc_str = f"{summary.get('binary_roc_auc'):.3f}" if summary.get("binary_roc_auc") is not None else "N/A"
    f1_str = f"{summary.get('macro_f1'):.3f}" if summary.get("macro_f1") is not None else "N/A"

    fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    sns.heatmap(binary_df, annot=True, fmt=".0f", cmap="Blues", ax=ax)
    ax.set_title(f"{subset_dir.name}\nMacro F1: {f1_str} | ROC-AUC: {auc_str}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    break  # remove this break to loop over all subsets
else:
    print("subset_dirs is empty")


In [ ]:
import ast
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
#hierarchical_optuna_bmr_multilabel_g2_20251216101932
results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/multilabel_optuna_bmr_stacking_fix_20260122085344/")
subset_dirs = sorted(
    d for d in results_root.iterdir()
    if d.is_dir() and d.name.startswith("rfecv_")
)

def load_confusion_df(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path, index_col=0) if path.exists() else None

def split_labels(series: pd.Series, delim: str = "|") -> list[list[str]]:
    def parse_entry(val: object) -> list[str]:
        if isinstance(val, list):
            return [str(x) for x in val if str(x)]
        as_str = "" if pd.isna(val) else str(val)
        stripped = as_str.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(as_str)
                if isinstance(parsed, (list, tuple)):
                    return [str(x) for x in parsed if str(x)]
            except Exception:
                pass
        return [x for x in as_str.split(delim) if x]
    return series.fillna("").apply(parse_entry).tolist()

def ranked_labels_from_proba(series: pd.Series) -> list[list[str]]:
    ranked: list[list[str]] = []
    for val in series.fillna(""):
        labels: list[str] = []
        try:
            parsed = ast.literal_eval(val) if isinstance(val, str) else val
            multiclass = parsed.get("multiclass") if isinstance(parsed, dict) else {}
            if multiclass:
                labels = [lbl for lbl, _ in sorted(multiclass.items(), key=lambda x: x[1], reverse=True)]
        except Exception:
            labels = []
        ranked.append(labels)
    return ranked

def precision_recall_at_k(true_lists: list[list[str]], ranked_preds: list[list[str]], k: int) -> tuple[float, float]:
    precisions: list[float] = []
    recalls: list[float] = []
    for true_labels, pred_labels in zip(true_lists, ranked_preds):
        topk = pred_labels[:k]
        hit = len(set(true_labels) & set(topk))
        precisions.append(hit / len(topk) if topk else 0.0)
        recalls.append(hit / len(true_labels) if true_labels else 0.0)
    return (
        sum(precisions) / len(precisions) if precisions else float("nan"),
        sum(recalls) / len(recalls) if recalls else float("nan"),
    )

print(subset_dirs)
for subset_dir in subset_dirs:
    preds_path = subset_dir / "hierarchical_predictions.csv"
    if not preds_path.exists():
        print(f"Could not find {preds_path}")
        continue

    preds = pd.read_csv(preds_path)
    negative_label = next(iter(set(preds["binary_pred"]) - {"POSITIVE"}), "NEGATIVE")

    # Binary confusion
    binary_df = load_confusion_df(subset_dir / "binary_confusion_matrix.csv")
    if binary_df is None:
        y_true_bin = preds["true_binary_label"].ne(negative_label).astype(int)
        y_pred_bin = preds["binary_pred"].eq("POSITIVE").astype(int)
        cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
        binary_df = pd.DataFrame(
            cm,
            index=[f"true_{negative_label}", "true_POSITIVE"],
            columns=[f"pred_{negative_label}", "pred_POSITIVE"],
        )
    for col in binary_df:
        binary_df[col] = binary_df[col].astype(int)

    # Detect multi-label run
    multilabel_metrics_path = subset_dir / "multilabel_metrics.json"
    is_multilabel = multilabel_metrics_path.exists() or (preds["hierarchical_pred"].astype(str).str.contains(r"\|").any())

    summary_path = subset_dir / "summary.json"
    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
    binary_suffix = (
        f"\nBinary macro ROC-AUC: {summary.get('binary_roc_auc'):.3f}"
        if summary.get("binary_roc_auc") is not None
        else ""
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(binary_df, annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
    axes[0].set_title(f"{subset_dir.name} – Binary{binary_suffix}")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("")

    if not is_multilabel:
        multiclass_df = load_confusion_df(subset_dir / "multiclass_confusion_matrix.csv")
        if multiclass_df is None:
            pos_mask = preds["true_binary_label"].ne(negative_label)
            multiclass_df = None
            if pos_mask.any():
                y_true_pos = preds.loc[pos_mask, "true_fenotipo"]
                y_pred_pos = preds.loc[pos_mask, "hierarchical_pred"]
                labels = sorted(set(y_true_pos) | set(y_pred_pos))
                cm = confusion_matrix(y_true_pos, y_pred_pos, labels=labels)
                multiclass_df = pd.DataFrame(
                    cm,
                    index=[f"true_{lbl}" for lbl in labels],
                    columns=[f"pred_{lbl}" for lbl in labels],
                )
        if multiclass_df is not None:
            sns.heatmap(multiclass_df, annot=True, fmt=".2f", cmap="Greens", ax=axes[1])
            axes[1].set_title(
                f"{subset_dir.name} – Multiclass "
                f"{'' if summary.get('multiclass_macro_auc') is None else f'(AUC {summary['multiclass_macro_auc']:.3f})'}"
            )
        else:
            axes[1].axis("off")
            axes[1].set_title(f"{subset_dir.name} – Multiclass")
    else:
        # Multi-label: show top label frequencies and metrics text
        pos_mask = preds["true_binary_label"].ne(negative_label)
        true_lists = split_labels(preds.loc[pos_mask, "true_fenotipo"])
        pred_lists = split_labels(preds.loc[pos_mask, "hierarchical_pred"])
        ranked_pred_lists = ranked_labels_from_proba(preds.loc[pos_mask, "hierarchical_proba"])
        true_flat = pd.Series([lbl for lst in true_lists for lbl in lst])
        pred_flat = pd.Series([lbl for lst in pred_lists for lbl in lst])
        top_true = true_flat.value_counts().head(10)
        top_pred = pred_flat.value_counts().head(10)
        freq_df = pd.DataFrame({"true": top_true, "pred": top_pred}).fillna(0).astype(int)
        freq_df.plot.bar(ax=axes[1], rot=90, title=f"{subset_dir.name} – Multi-label top-10 labels")
        axes[1].set_ylabel("count")

        ml_metrics = json.loads(multilabel_metrics_path.read_text()) if multilabel_metrics_path.exists() else {}

        precision_at_1_calc, recall_at_1_calc = precision_recall_at_k(true_lists, ranked_pred_lists, 1)
        precision_at_3_calc, recall_at_3_calc = precision_recall_at_k(true_lists, ranked_pred_lists, 3)

        precision_at_1_val = precision_at_1_calc if not pd.isna(precision_at_1_calc) else ml_metrics.get("precision_at_1", float("nan"))
        recall_at_1_val = recall_at_1_calc if not pd.isna(recall_at_1_calc) else ml_metrics.get("recall_at_1", float("nan"))
        precision_at_3_val = precision_at_3_calc if not pd.isna(precision_at_3_calc) else ml_metrics.get("precision_at_3", float("nan"))
        recall_at_3_val = recall_at_3_calc if not pd.isna(recall_at_3_calc) else ml_metrics.get("recall_at_3", float("nan"))

        text = "\n".join(
            [
                f"micro-F1: {ml_metrics.get('micro_f1', float('nan')):.3f}",
                f"macro-F1: {ml_metrics.get('macro_f1', float('nan')):.3f}",
                f"exact match: {ml_metrics.get('exact_match_ratio', float('nan')):.3f}",
                f"coverage: {ml_metrics.get('coverage', float('nan')):.3f}",
                f"inclusion_any_true: {ml_metrics.get('inclusion_any_true', float('nan')):.3f}",
                f"avg labels pred: {ml_metrics.get('avg_labels_predicted', float('nan')):.2f}",
                f"precision_at_top1: {precision_at_1_val:.2f}",
                f"recall_at_top1: {recall_at_1_val:.2f}",
                f"precision_at_top3: {precision_at_3_val:.2f}",
                f"recall_at_top3: {recall_at_3_val:.2f}",
            ]
        )
        axes[1].text(1.05, 0.5, text, transform=axes[1].transAxes, va="center")

    plt.tight_layout()
    plt.show()
    break
else:
    print("subset_dirs is empty")


In [ ]:
summary

In [ ]:
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid")

results_root = Path("/data/ucct/bi/scratch_tmp/bi/TESTS/pmata_mepram_tests/bmr_phenotest/outputs/hierarchical_optuna_bmr_multilabel_20251126113557/")
subset_dirs = sorted(d for d in results_root.iterdir() if d.is_dir() and d.name.startswith("rfecv_"))
if not subset_dirs:
    raise SystemExit("No subset dirs found")
subset_dir = subset_dirs[0]  # pick the first; change if needed

preds = pd.read_csv(subset_dir / "hierarchical_predictions.csv")
negative_label = next(iter(set(preds["binary_pred"]) - {"POSITIVE"}), "NEGATIVE")

# --- Binary confusion
binary_df = pd.read_csv(subset_dir / "binary_confusion_matrix.csv", index_col=0) if (subset_dir / "binary_confusion_matrix.csv").exists() else None
if binary_df is None:
    y_true_bin = preds["true_binary_label"].ne(negative_label).astype(int)
    y_pred_bin = preds["binary_pred"].eq("POSITIVE").astype(int)
    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    binary_df = pd.DataFrame(cm,
        index=[f"true_{negative_label}", "true_POSITIVE"],
        columns=[f"pred_{negative_label}", "pred_POSITIVE"],
    )
binary_df = binary_df.astype(int)

# --- Multi-label prep (positive samples only)
pos_mask = preds["true_binary_label"].ne(negative_label)
true_lists = preds.loc[pos_mask, "true_fenotipo"].fillna("").apply(lambda s: [x for x in str(s).split("|") if x]).tolist()
pred_lists = preds.loc[pos_mask, "hierarchical_pred"].fillna("").apply(lambda s: [x for x in str(s).split("|") if x]).tolist()

# Per-label recall
true_counts = Counter(lbl for lst in true_lists for lbl in lst)
correct_counts = Counter()
for tl, pl in zip(true_lists, pred_lists):
    ps = set(pl)
    for lbl in tl:
        if lbl in ps:
            correct_counts[lbl] += 1
recall_df = pd.DataFrame([
    {"label": lbl, "recall": correct_counts.get(lbl, 0) / cnt, "support": cnt}
    for lbl, cnt in true_counts.items()
]).sort_values("recall", ascending=False)

# Top labels (true vs pred)
true_flat = Counter(lbl for lst in true_lists for lbl in lst)
pred_flat = Counter(lbl for lst in pred_lists for lbl in lst)
top_true = true_flat.most_common(7)
top_pred = pred_flat.most_common(7)
top_labels = list({lbl for lbl, _ in top_true} | {lbl for lbl, _ in top_pred})
top_df = pd.DataFrame({
    "true": {lbl: true_flat.get(lbl, 0) for lbl in top_labels},
    "pred": {lbl: pred_flat.get(lbl, 0) for lbl in top_labels},
}).reindex(index=sorted(top_labels))

# Label count per patient + coverage/hit-any
true_counts_per = [len(x) for x in true_lists]
pred_counts_per = [len(x) for x in pred_lists]
coverage = sum(c > 0 for c in pred_counts_per) / len(pred_counts_per) if pred_counts_per else 0.0
hit_any = sum(len(set(t) & set(p)) > 0 for t, p in zip(true_lists, pred_lists)) / len(true_lists) if true_lists else 0.0

# Optional saved metrics
ml_metrics = {}
ml_path = subset_dir / "multilabel_metrics.json"
if ml_path.exists():
    ml_metrics = json.loads(ml_path.read_text())

# --- Plotting (4 panels)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.heatmap(binary_df, annot=True, fmt="d", cmap="Blues", ax=axes[0, 0])
axes[0, 0].set_title(f"{subset_dir.name} – Binary gate")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("")

sns.barplot(
    data=recall_df,
    x="recall",
    y="label",
    order=recall_df["label"].tolist(),
    palette="Greens_r",
    ax=axes[0, 1],
)
axes[0, 1].set_title("Per-label recall (multi-label)")
axes[0, 1].set_xlim(0, 1)

top_df.plot.bar(ax=axes[1, 0], rot=45, title="Top labels: true vs predicted")
axes[1, 0].set_ylabel("count")

axes[1, 1].hist(true_counts_per, bins=range(0, max(true_counts_per + pred_counts_per + [1]) + 1), alpha=0.6, label="true")
axes[1, 1].hist(pred_counts_per, bins=range(0, max(true_counts_per + pred_counts_per + [1]) + 1), alpha=0.6, label="pred")
axes[1, 1].set_title(f"Labels per patient\ncoverage(any pred): {coverage:.2f} | hit(any correct): {hit_any:.2f}")
axes[1, 1].set_xlabel("# labels")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

if ml_metrics:
    print("Saved multi-label metrics:")
    for k, v in ml_metrics.items():
        print(f"  {k}: {v}")


In [ ]:
foco_map = {1.0: 'pulmonar',
 2.0: 'intraabdominal',
 3.0: 'biliar',
 4.0: 'urinario',
 5.0: 'cardiovascular',
 6.0: 'piel',
 7.0: 'sistema nervioso central',
 8.0: 'catéter venoso',
 9.0: 'vías altas respiratorias',
 10.0: 'osteoarticular',
 11.0: 'genital',
 12.0: 'desconocido'}

## Visualisation setup

Load plotting libraries and helper metadata that support the upcoming exploratory analyses.

In [ ]:
sintom_dict = tbl_codes2names[tbl_codes2names["name"].str.contains("síntoma ")][["value", "name"]].to_dict(orient="records")
sintom_dict = {"sintoma_"+str(float(d["value"])): "sintoma_" + d["name"].replace("síntoma | ", "") for d in sintom_dict}
sintom_map = {}
for k,v in sintom_dict.items():
    sintom_map[k] = v
    sintom_map[k+"_categorico"] = v+"_categorico"
sintom_map

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
import copy

def correlation_ratio(categories, measurements):
    """Correlation ratio (eta squared) between categorical target and numerical feature"""
    categories = np.array(categories)
    measurements = np.array(measurements)
    cat_groups = [measurements[categories == cat] for cat in np.unique(categories)]
    means = [np.mean(g) for g in cat_groups if len(g) > 0]
    n = len(measurements)
    grand_mean = np.mean(measurements)
    ss_between = sum(len(g) * (m - grand_mean) ** 2 for g, m in zip(cat_groups, means))
    ss_total = sum((measurements - grand_mean) ** 2)
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else 0

# -------------------------------------
target = "resultado_hemo"
df_to_plot = df_expanded.copy().rename(columns=sintom_map)
df = copy.deepcopy(df_to_plot.drop(["freq_bac_foco", "freq_bacteria", "infected_yes_no", "person_id"], axis=1))
df = df[~df["resultado_hemo"].isin(["Enterococcus", "_Fungi", "_Other bacteria", '_Virus'])]

antibmap = {x["value"]:x["name"].split("|")[1].strip() for x in tbl_codes2names[tbl_codes2names["variable"] == "antimicrobiano_previo"][["value", "name"]].to_dict(orient="records")}
df["ultimo_antib"] = df["ultimo_antib"].map(antibmap)
df["ultimo_antib"] = LabelEncoder().fit_transform(df["ultimo_antib"])
#df = pd.get_dummies(df, columns=["ultimo_antib"], drop_first=True)
# Get categories directly
classes = df[target].unique()
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target]
"""df = pd.get_dummies(df, columns=["ultimo_antib"], drop_first=True)
df = df.rename(columns=lambda c: c.replace('ultimo_antib_', ''))
numeric_cols = df.drop(target, axis=1).columns"""
# Prepare a dataframe to store correlations per class and feature
corr_matrix = pd.DataFrame(index=classes, columns=numeric_cols, dtype=float)

for cls in classes:
    # binary target: this class vs the rest
    y_binary = (df[target] == cls).astype(int)
    for col in numeric_cols:
        corr_matrix.loc[cls, col] = correlation_ratio(y_binary, df[col].values)

# Optional: filter weak correlations
filtered = corr_matrix.loc[:, corr_matrix.max(axis=0) > 0.1]

# -------------------------------------
# 🔹 Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(filtered, annot=False, cmap="viridis", cbar_kws={"label": "η (class-wise correlation)"})
plt.title(f"Ultimo_antib correlation ratio (η) by etiology. Showing only those with η > 0.02")
plt.xlabel("Feature")
plt.ylabel("Bacteria / Etiology class")
plt.tight_layout()
plt.show()


## Subset for etiology modelling

Create a reduced dataframe focused on pre-admission events (infections, colonisations) paired with hemoculture outcomes.

In [ ]:
df_hemo_merged = pd.merge(df_pacientes, hemo_urg_pivoted[["person_id", "resultado_hemo"]], on= ['person_id'], how= 'left')

orig_cols = df_hemo_merged.columns.tolist()
df_hemo_merged = df_hemo_merged.merge(tbl_infecciones_complete, on= ['person_id'], how= 'left')
df_hemo_merged = df_hemo_merged.merge(colo_prev_pivoted, on = ['person_id'], how= 'left')
new_cols = [c for c in df_hemo_merged.columns if c not in orig_cols]
df_hemo_merged[new_cols] = df_hemo_merged[new_cols].fillna(0) # Not all patients have previous infections/colonizations

df_hemo_merged = df_hemo_merged.merge(tbl_sepsis, on= ['person_id', 'fecha_ingreso_urgencias'], how= 'left')

In [ ]:
df_expanded = df_hemo_merged.explode("resultado_hemo").reset_index(drop=True)
counts = df_expanded["person_id"].value_counts()
df_expanded["weight"] = df_expanded["person_id"].apply(lambda x: 1 / counts[x])
df_expanded["co_infection"] = np.where(df_expanded["weight"] < 1, 1, 0)

In [ ]:
df_expanded.to_csv(os.path.join(save_location, "df_hemo_merged_v2.csv"), index=False)

## Generate automated EDA report

Reload the merged dataset and render an HTML profiling report with `ydata_profiling`.

In [ ]:
df_merged = pd.read_csv(os.path.join(save_location, "df_merged_full.csv"))

In [ ]:
profile = ProfileReport(df_merged, title="MePRAM EDA report")
profile.to_notebook_iframe()
profile.to_file(os.path.join(save_location, "df_merge_report.html"))

## Visualise missingness patterns

Plot heatmaps that highlight columns with substantial fractions of missing data to guide imputation strategies.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def insert_linebreak(string, lengLabel=10):
    return '\n'.join(string[i:i+lengLabel] for i in range(0, len(string), lengLabel))
merged_df = pd.read_csv(os.path.join(save_location, "df_merged.csv"))
prev_num = 0

def plot_missing_columns(subset_df, title=f"Missing Data Matrix for columns 0 to 171"):
    missing_df = subset_df.isna()

    missing_df.columns = subset_df.columns
    missing_percent = missing_df.mean() * 100
    columns_with_percent = [
        f"$\\bf{{{col}}}$ ({missing_percent[col]:.2f}%)" if missing_percent[col] > 30 else f"{col} ({missing_percent[col]:.2f}%)"
        for col in subset_df.columns
    ]
    plt.figure(figsize=(24, 6))
    ax = sns.heatmap(missing_df, vmin=0, vmax=1, cbar=False,
                xticklabels=columns_with_percent)
    ax.tick_params(axis='x', which='minor', length=40)
    plt.title(title)
    plt.xlabel("Columns (% Missing)")
    plt.ylabel("Rows", rotation=90)
    plt.xticks(rotation=90)
    plt.show()


plot_missing_columns(merged_df)

missing_df = merged_df.isna()
missing_df.columns = merged_df.columns
missing_percent = missing_df.mean() * 100
print([idx for idx,x in enumerate(missing_percent) if x > 30])
dangerous_df = merged_df.iloc[:, [idx for idx,x in enumerate(missing_percent) if x > 30]]
print(dangerous_df)
plot_missing_columns(dangerous_df, f"Missing Data Matrix of {len(dangerous_df.columns)} columns with > 30% NAs")

## Principal component analysis for feature exploration

Scale features with `MinMaxScaler`, fit 2D/3D PCA components, and visualise them interactively.

In [ ]:
TARGET_VARIABLE = "sepsis"

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

target = df_merged[TARGET_VARIABLE]
data = df_merged.drop(columns= [TARGET_VARIABLE])

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

explained_variance = pca.explained_variance_ratio_
print(f"Varianza explicada por cada componente: {explained_variance}")
print(f"Varianza total explicada: {sum(explained_variance)}")

plt.figure(figsize=(8, 6))
plt.scatter(pca_data[:, 0], pca_data[:, 1], c=target, cmap='viridis', alpha=0.7)
plt.title("PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()


In [ ]:
import plotly.express as px 

pca_3d = PCA(n_components=3)
pca_data_3d = pca_3d.fit_transform(scaled_data)

pca_df = pd.DataFrame(pca_data_3d, columns=["PC1", "PC2" ,"PC3"])
pca_df[TARGET_VARIABLE] = target

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color = TARGET_VARIABLE,
    title = "PCA - 3D Visualization",
    labels= {TARGET_VARIABLE},
    color_continuous_scale="Viridis", 
    opacity=0.7  
)
fig.update_traces(marker=dict(size=5))  
fig.update_layout(scene=dict(
    xaxis_title="PC 1",
    yaxis_title="PC 2",
    zaxis_title="PC 3"
))
fig.show()

In [ ]:
processed_df_copy = df_merged

target_copy = processed_df_copy[TARGET_VARIABLE]
scaler = MinMaxScaler()
X_preprocessed = pd.DataFrame(scaler.fit_transform(processed_df_copy.drop(columns=[TARGET_VARIABLE])))
X_preprocessed[TARGET_VARIABLE] = target_copy

target_palette = {0: "blue", 1: "red"}
row_colors = X_preprocessed[TARGET_VARIABLE].map(target_palette)
X_preprocessed = X_preprocessed.dropna() 
sns.clustermap(
    X_preprocessed.drop(columns=[TARGET_VARIABLE]), 
    cmap="coolwarm",
    row_colors=row_colors,
    figsize=(30, 60),
    annot=False,
    col_cluster=False
)

plt.title("Heatmap con Clustering y Anotación por Target", pad=100)
plt.show()